## Projet Santé Mentale des Adolescents

In [1670]:
# Dépendances du notebook
%pip install openpyxl==3.1.3 pandas==3.0.2 s3fs==2026.3.0 -q

Note: you may need to restart the kernel to use updated packages.


## Importation des packages nécessaires

In [1671]:
import pandas as pd
import os
import openpyxl
from openpyxl import *
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.utils import *
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl import Workbook   
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl import load_workbook 
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill
from openpyxl.chart import BarChart, Reference
from openpyxl.styles import Font, Border, Side
from openpyxl.styles import Alignment
from openpyxl.chart.label import DataLabelList                                                                                                                                                      
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.utils import quote_sheetname
from openpyxl.utils.cell import coordinate_from_string, column_index_from_string
from openpyxl.worksheet.worksheet import Worksheet
from openpyxl.styles import Alignment, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation
import pandas as pd
from PIL import Image

print(openpyxl.__version__)


3.1.3


### Importation des données - Santé Mentale

Après l'importation, on inspecte les types de données présents.

In [1672]:
df = pd.read_csv('https://minio.lab.sspcloud.fr/nerojeni10/DATA_PROJET_SMA/Teen_Mental_Health_Dataset.csv')

df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       1200 non-null   int64  
 1   gender                    1200 non-null   str    
 2   daily_social_media_hours  1200 non-null   float64
 3   platform_usage            1200 non-null   str    
 4   sleep_hours               1200 non-null   float64
 5   screen_time_before_sleep  1200 non-null   float64
 6   academic_performance      1200 non-null   float64
 7   physical_activity         1200 non-null   float64
 8   social_interaction_level  1200 non-null   str    
 9   stress_level              1200 non-null   int64  
 10  anxiety_level             1200 non-null   int64  
 11  addiction_level           1200 non-null   int64  
 12  depression_label          1200 non-null   int64  
dtypes: float64(5), int64(5), str(3)
memory usage: 140.4 KB


,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0


### Inspecter la présence des valeurs manquantes

In [1673]:
df.isnull().sum()

age                         0
gender                      0
daily_social_media_hours    0
platform_usage              0
sleep_hours                 0
screen_time_before_sleep    0
academic_performance        0
physical_activity           0
social_interaction_level    0
stress_level                0
anxiety_level               0
addiction_level             0
depression_label            0
dtype: int64

***Il n'y a aucune valeur manquante dans ce jeux de données***

## Remplissage du fichier

Pour remplir le fichier, on allons créer plusieurs feuilles composées des données nécessaires à la création des indicateurs

In [1674]:
path_file = "../template/Projet_ODD_SIVARAJAH.xlsx"

# Recréer un fichier propre sans feuille parasite
try:
    wb = load_workbook(path_file)  # noqa: F405
except Exception:
    wb = Workbook()


# Créer un vrai fichier Excel vide si inexistant
if not os.path.exists(path_file):
    wb = Workbook()
    wb.save(path_file)


# Ajouter la feuille DATA
with pd.ExcelWriter(path_file, mode="a", if_sheet_exists="replace") as writer:
    df.to_excel(writer, sheet_name='DATA', index=False)



print("Feuilles présentes :", load_workbook(path_file).sheetnames)

Feuilles présentes : ['Sheet', 'DATA']


## Création de la feuille CALC

Sur cette feuille, on  aura les valeurs distinctes pour chaque variable, afin de réaliser des groupes plus tard et de réaliser des agrégations dessus.

In [1675]:
# # Chargement du fichier en mémoire
# wb = load_workbook(path_file)

# # Créer la feuille CALC si elle n'existe pas
# if "CALC" not in wb.sheetnames:
#     ws = wb.create_sheet("CALC")
# else:
#     ws = wb["CALC"]

### Création des variables distinctes

In [1676]:
# from openpyxl.utils import FORMULAE
# "UNIQUE" in FORMULAE

# # ws["A1"]="Genres distincts"
# formula = "=_xlfn.UNIQUE(DATA!B2:B)"
# ws["A1"]=ArrayFormula("A1:A", formula)

# # # ws["A3"]="Ages distincts"
# # # ws["A3"]= "=_xlfn.UNIQUE(DATA!A2:A)"

# # # ws["A5"]="Plateformes distincts"
# # # ws["A5"]= "=_xlfn.UNIQUE(DATA!D2:D)"

# # # ws["A7"]="Social Interactions"
# # # ws["A7"]= "=_xlfn.UNIQUE(DATA!I2:I)"


# # wb.save(path_file)


## Création des indicateurs

In [1677]:
# Chargement du fichier en mémoire
wb = load_workbook(path_file)


# Supprimer la feuille vide par défaut si elle existe
if "Sheet" in wb.sheetnames:
    del wb["Sheet"]

wb.save(path_file)

# Créer la feuille Indicateurs si elle n'existe pas
if "Indicateurs" not in wb.sheetnames:
    ws = wb.create_sheet("Indicateurs")
else:
    ws = wb["Indicateurs"]

# Ajout des formules
# 1. Nombre de filles dépressives
ws['A1'] = "Nombre de filles dépressives"
ws['B1'] = '=COUNTIFS(DATA!M:M,1,DATA!B:B,"female")'

# 2. Nombre de garçons dépressifs
ws['A2'] = "Nombre de garçons dépressifs"
ws['B2'] = '=COUNTIFS(DATA!M:M,1,DATA!B:B,"male")'

# 3. Niveau d'addiction moyen chez les filles
ws['A3'] = "Niveau d'addiction moyen chez les filles"
ws['B3'] = '=AVERAGEIF(DATA!B:B,"female",DATA!L:L)'

# 4. Niveau d'addiction moyen chez les garçons
ws['A4'] = "Niveau d'addiction moyen chez les garçons"
ws['B4'] = '=AVERAGEIF(DATA!B:B,"male",DATA!L:L)'


# Création d'une nouvelle feuille, TCD (Tableau croisé dynamique)

Sur cette feuille apparaîtrant les indicateurs qui sont groupés selon différents critères comme l'âge ou le genre. 

Pour plus de simplicité, la réalisation de ces groupes et des agrégations nécessaires j'utilise la bibliothèque pandas et les résultats sont par la suite transcris dans les feuilles. 

Afin d'automatiser l'écriture des données, et d'éviter le chevauchement des résultats une fonction est crée pour CALCuler automatiquement la cellule dans la  quelle on commencera à écrire les données.

In [1678]:
def write_table(ws, df, start_row, title=None, space=3, padding=2):
    """
    Écrit un DataFrame dans une feuille Excel OpenPyXL à partir d'une ligne donnée.

    La fonction ajoute éventuellement un titre, écrit les en-têtes de colonnes
    puis les données du DataFrame. Elle retourne ensuite la première ligne
    disponible pour écrire un nouveau tableau en laissant un nombre de lignes
    vides configurable.

    Args:
        ws: Feuille OpenPyXL cible.
        df: DataFrame à écrire.
        start_row (int): Ligne de départ.
        title (str, optional): Titre du tableau.
        space (int, optional): Nombre de lignes vides à laisser après le tableau.

    Returns:
        int: Numéro de la prochaine ligne disponible.

    Examples:
    >>> start_row = 1
    >>> start_row = write_table(ws, tcd1, start_row,
    ...                         "Dépression selon l'âge")
    >>> start_row = write_table(ws, tcd2, start_row,
    ...                         "Addiction moyenne selon l'âge et le genre")
    """

    # En-têtes
    if title:
        ws.cell(row=start_row, column=1, value=title)
        start_row += 1

    # En-têtes + ajustement largeur colonnes
    for col_idx, header in enumerate(df.columns, start=1):
        ws.cell(row=start_row, column=col_idx, value=header)

        col_letter = get_column_letter(col_idx)
        width = len(str(header)) + padding

        # On conserve la plus grande largeur si la colonne existe déjà
        current_width = ws.column_dimensions[col_letter].width
        if current_width is None or width > current_width:
            ws.column_dimensions[col_letter].width = width

    # Données
    for i, row in df.iterrows():
        for col_idx, value in enumerate(row, start=1):
            ws.cell(
                row=start_row + i + 1,
                column=col_idx,
                value=value
            )

    # Ligne de départ du tableau suivant
    return start_row + len(df) + space + 1

### Création de la feuille TCD si elle n'existe pas

In [1679]:
if "TCD" not in wb.sheetnames:
    ws_tcd = wb.create_sheet("TCD")
else:
    ws_tcd = wb["TCD"]


### Création des indicateurs et écriture des données avec la fonction créée

### Création d'une matrice de corrélation

Pour étudier les liens entre les différents variables de ce jeux de données

In [1680]:
# Encoder les variables catégorielles en numérique
df["gender_num"] = df["gender"].map({"male": 0, "female": 1})
df["social_num"] = df["social_interaction_level"].map({"low": 0, "medium": 1, "high": 2})

# Sélectionner les colonnes numériques
cols_corr = ["age", "sleep_hours", "daily_social_media_hours",
             "academic_performance", "physical_activity",
             "social_num", "stress_level", "anxiety_level",
             "addiction_level", "depression_label"]

# Matrice de corrélation
corr = df[cols_corr].corr().round(2)

corr_reset = corr.reset_index()
corr_reset.columns = ["Variable"] + cols_corr

# Création d'une nouvelle feuille pour réaliser le tableau de corrélation
wb = load_workbook(path_file)

if "Correlations" not in wb.sheetnames:
    ws_corr = wb.create_sheet("Correlations")
else:
    ws_corr = wb["Correlations"]

write_table(
    ws_corr,
    corr_reset,
    start_row=1,
    title="Matrice de corrélation",
    space=0
)

# Sauvegarde du fichier
wb.save(path_file)
wb.close()
print("Feuilles presentes :", load_workbook(path_file).sheetnames)

Feuilles presentes : ['DATA', 'Correlations']


In [1681]:

cols_to_calculate = ['age', 'gender', 'platform_usage', 'social_interaction_level']
len_dict ={}
for col in cols_to_calculate:
    len_dict[f"len_{col}"] = len(df[col].unique())+1 
print(f'{len_dict}')



{'len_age': 8, 'len_gender': 3, 'len_platform_usage': 4, 'len_social_interaction_level': 4}


In [1682]:
from openpyxl import load_workbook
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.worksheet.table import Table, TableStyleInfo

wb = load_workbook(path_file)

if "CALC" not in wb.sheetnames:
    ws_calc = wb.create_sheet("CALC")
else:
    ws_calc = wb["CALC"]

style = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False
)

# Genres - colonne A
formula = "=_xlfn.UNIQUE(DATA!B:B)"
ws_calc['A1'] = ArrayFormula(
    f"A1:A{len_dict['len_gender']}",
    formula
)
table_gender = Table(displayName="tblGenres", ref=f"A1:A{len_dict['len_gender']}")
table_gender.tableStyleInfo = style
table_gender.hasHeader = False
ws_calc.add_table(table_gender)

# Ages - colonne C
formula = "=_xlfn.UNIQUE(DATA!A:A)"
ws_calc['C1'] = ArrayFormula(
    f"C1:C{len_dict['len_age']}",
    formula
)
table_age = Table(displayName="tblAges", ref=f"C1:C{len_dict['len_age']}")
table_age.tableStyleInfo = style
table_age.hasHeader = False
ws_calc.add_table(table_age)

# Plateformes - colonne E
formula = "=_xlfn.UNIQUE(DATA!D:D)"
ws_calc['E1'] = ArrayFormula(
    f"E1:E{len_dict['len_platform_usage']}",
    formula
)
table_platform = Table(displayName="tblPlateformes", ref=f"E1:E{len_dict['len_platform_usage']}")
table_platform.tableStyleInfo = style
table_platform.hasHeader = False
ws_calc.add_table(table_platform)

# Interactions sociales - colonne G
formula = "=_xlfn.UNIQUE(DATA!I:I)"
ws_calc['G1'] = ArrayFormula(
    f"G1:G{len_dict['len_social_interaction_level']}",
    formula
)
table_interaction = Table(displayName="tblInteractions", ref=f"G1:G{len_dict['len_social_interaction_level']}")
table_interaction.tableStyleInfo = style
table_interaction.hasHeader = False
ws_calc.add_table(table_interaction)

wb.save(path_file)
wb.close()
print("Feuilles presentes :", load_workbook(path_file).sheetnames)

Feuilles presentes : ['DATA', 'Correlations', 'CALC']


/opt/python/lib/python3.13/site-packages/openpyxl/worksheet/_writer.py:274: UserWarning: File may not be readable: column headings must be strings.
  warn("File may not be readable: column headings must be strings.")


# Création du dashboard

## Création des filtres

Maintenant qu'on a les indicateurs uniques, on peut les utiliser pour la création de filtres.

In [1683]:
from openpyxl.styles import Alignment, PatternFill, Font, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation

def add_filter(worksheet, start_col, filter_row, title_text, 
               data_source_col, len_data, helper_col, default_value='Tous', 
               title_color='FF003D6B', value_color='FFFFFF'):
    """
    Crée un filtre horizontal avec titre et valeur côte à côte.
    
    Le filtre génère une colonne cachée (helper_col) pour inclure l'option "Tous"
    sans modifier la feuille CALC d'origine. Les titres et valeurs ont des couleurs différentes.
    
    Parameters
    ----------
    worksheet : openpyxl.worksheet.worksheet.Worksheet
        La feuille de travail où ajouter le filtre.
    start_col : str
        Colonne de départ pour le titre (ex: 'C'). 
        La valeur sera dans la colonne suivante (ex: 'D').
    filter_row : int
        La ligne où placer le filtre (ex: 3).
    title_text : str
        Le texte du titre du filtre (ex: 'Âge', 'Genre').
    data_source_col : str
        La colonne source dans CALC (ex: 'C', 'A', 'E').
    len_data : int
        Le nombre de valeurs uniques (ex: len_dict['len_age']).
    helper_col : str
        La colonne cachée pour stocker les données (ex: 'AA', 'AB').
    default_value : str, optional
        La valeur par défaut affichée. Par défaut, 'Tous'.
    title_color : str, optional
        Couleur hexadécimale du titre (ex: 'FF003D6B'). Par défaut, teal foncé.
    value_color : str, optional
        Couleur hexadécimale de la valeur (ex: 'FFFFFF'). Par défaut, teal clair.
    
    Returns
    -------
    None
        Modifie la feuille en place.
    
    Examples
    --------
    >>> add_filter(ws, 'C', 3, 'Âge', 'C', 8, 'AB')
    # Crée un filtre "Âge" avec titre teal foncé et valeur teal clair
    
    Notes
    -----
    - Le titre est placé dans start_col, la valeur dans la colonne suivante.
    - L'option "Tous" est automatiquement ajoutée sans modifier CALC.
    - Les colonnes helper sont masquées du tableau de bord.
    """
    
    # Colonne de la valeur (juste après le titre)
    value_col = chr(ord(start_col) + 1)
    
    # Remplissage pour titre et valeur
    title_fill = PatternFill(start_color=title_color, end_color=title_color, fill_type='solid')
    value_fill = PatternFill(start_color=value_color, end_color=value_color, fill_type='solid')
    
    alignment = Alignment(horizontal='center', vertical='center')
    border = Border(
        left=Side(style='thin'), right=Side(style='thin'),
        top=Side(style='thin'), bottom=Side(style='thin')
    )
    title_font = Font(bold=True, color='FFFFFF', size=10)
    value_font = Font(bold=True, color='000000', size=10)  # Texte noir pour le clair
    
    # ========== TITRE DU FILTRE ==========
    title_cell = worksheet[f'{start_col}{filter_row}']
    title_cell.value = title_text
    title_cell.alignment = alignment
    title_cell.fill = title_fill
    title_cell.border = border
    title_cell.font = title_font
    worksheet.column_dimensions[start_col].width = 12
    
    # ========== CELLULE DE VALEUR (couleur plus claire) ==========
    value_cell = worksheet[f'{value_col}{filter_row}']
    value_cell.value = default_value
    value_cell.alignment = alignment
    value_cell.fill = value_fill
    value_cell.border = border
    value_cell.font = value_font
    worksheet.column_dimensions[value_col].width = 12
    
    # ========== COLONNE HELPER (CACHÉE) ==========
    worksheet[f'{helper_col}1'] = 'Tous'
    
    for i in range(2, len_data + 1):
        worksheet[f'{helper_col}{i}'] = f'=CALC!{data_source_col}{i}'
    
    worksheet.column_dimensions[helper_col].hidden = True
    
    # ========== VALIDATION DE DONNÉES ==========
    formula = f"=${helper_col}$1:${helper_col}${len_data}"
    
    dv = DataValidation(type='list', formula1=formula, allow_blank=False)
    dv.error = 'Sélectionnez une valeur valide'
    dv.errorTitle = 'Entrée invalide'
    dv.prompt = f'Sélectionnez un {title_text.lower()}'
    dv.promptTitle = 'Filtrer'
    
    worksheet.add_data_validation(dv)
    dv.add(f'{value_col}{filter_row}')
    
    print(f"✅ Filtre '{title_text}' créé en {start_col}{filter_row}:{value_col}{filter_row}")

# ============================================================================
# PAGE 1 - TITRE BLEU + FILTRES AVEC DISTINCTION COULEUR
# ============================================================================

print("\n📊 Création de TDB1 avec titre bleu et filtres contrastés...\n")

# Nettoyer si existe déjà
if "TDB1" in wb.sheetnames:
    del wb["TDB1"]
TDB1 = wb.create_sheet("TDB1", 0)
TDB1.sheet_view.showGridLines = False

# ========== TITRE PRINCIPAL EN BLEU FONCÉ ==========
title_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type='solid')  # Bleu foncé
title_font = Font(name='Calibri', size=14, bold=True, color='FFFFFF')
title_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

TDB1.merge_cells('A1:O1')
TDB1['A1'] = 'Impact des réseaux sociaux sur la santé mentale'
TDB1['A1'].fill = title_fill
TDB1['A1'].font = title_font
TDB1['A1'].alignment = title_alignment
TDB1.row_dimensions[1].height = 30

# ========== LIGNE DE FILTRES ==========
TDB1['A3'] = 'Filtres :'
TDB1['A3'].font = Font(size=10, bold=True)
TDB1['A3'].alignment = Alignment(horizontal='left', vertical='center')

# FILTRES HORIZONTAUX
# Format: add_filter(ws, start_col_titre, row, titre, col_CALC, len_data, col_helper, 
#                    default, title_color, value_color)

# Teal foncé pour titre (#008080), Teal clair pour valeur (#B3E5E0)
add_filter(TDB1, 'C', 3, 'Âge', 'C', len_dict['len_age'], 'AA', 
           title_color='FF003D6B', value_color='FFFFFF')

add_filter(TDB1, 'F', 3, 'Genre', 'A', len_dict['len_gender'], 'AB',
           title_color='FF003D6B', value_color='FFFFFF')

add_filter(TDB1, 'I', 3, 'Plateforme', 'E', len_dict['len_platform_usage'], 'AC',
           title_color='FF003D6B', value_color='FFFFFF')

add_filter(TDB1, 'L', 3, 'Interaction', 'G', len_dict['len_social_interaction_level'], 'AD',
           title_color='FF003D6B', value_color='FFFFFF')

# Définir la hauteur de la ligne des filtres
TDB1.row_dimensions[3].height = 25

wb.save(path_file)
print("\n✅ TDB1 créée avec succès !")
print(f"📋 Feuilles présentes : {wb.sheetnames}\n")
wb.close()


📊 Création de TDB1 avec titre bleu et filtres contrastés...

✅ Filtre 'Âge' créé en C3:D3
✅ Filtre 'Genre' créé en F3:G3
✅ Filtre 'Plateforme' créé en I3:J3
✅ Filtre 'Interaction' créé en L3:M3

✅ TDB1 créée avec succès !
📋 Feuilles présentes : ['TDB1', 'DATA', 'Correlations', 'CALC']



# Création des tableaux groupés

Ces tableaux permettront de créer les graphiques reposant sur plusieurs critères par exemple.

In [1684]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

wb = load_workbook(path_file)

# ========== STYLES ==========
title_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")  # Bleu foncé
title_font = Font(bold=True, size=12, color="FFFFFF")
header_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")  # Teal foncé
header_font = Font(color="FFFFFF", bold=True, size=10)
row_header_fill = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")  # Teal clair
row_header_font = Font(bold=True, color="000000", size=10)
border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)
center_align = Alignment(horizontal="center", vertical="center")

# ========== CRÉATION DE LA FEUILLE ==========
if "TCD" in wb.sheetnames:
    del wb["TCD"]
ws_tcd = wb.create_sheet("TCD")

# =================================================================
# TABLEAU TCD DYNAMIQUE : Dépression par Genre et Âge
# (N'affiche que les lignes/colonnes correspondant aux filtres)
# =================================================================

# --- TITRE ---
ws_tcd['A1'] = "Dépression par Genre et Âge"
ws_tcd['A1'].font = title_font
ws_tcd['A1'].fill = title_fill
ws_tcd['A1'].alignment = center_align
ws_tcd.merge_cells('A1:J1')
ws_tcd.row_dimensions[1].height = 25

# --- EN-TÊTE COLONNES (Âges depuis CALC!C) - DYNAMIQUES ---
ws_tcd['A2'] = "Genre"
ws_tcd['A2'].fill = header_fill
ws_tcd['A2'].font = header_font
ws_tcd['A2'].border = border
ws_tcd['A2'].alignment = center_align
ws_tcd.column_dimensions['A'].width = 15

# Insérer tous les âges en en-tête de colonne (CONDITIONNELLEMENT)
for col_idx in range(1, len_dict['len_age']):
    col_letter = get_column_letter(col_idx + 1)
    
    # N'affiche l'âge QUE si :
    # - Le filtre Âge = "Tous" (affiche tous les âges)
    # - OU l'âge de la colonne = le filtre Âge
    ws_tcd[f'{col_letter}2'] = (
        f'=IF(OR(TDB1!$D$3="Tous", IFERROR(INDEX(CALC!$C$2:$C$100,{col_idx}),)=TDB1!$D$3), '
        f'IFERROR(INDEX(CALC!$C$2:$C$100,{col_idx}),""), "")'
    )
    ws_tcd[f'{col_letter}2'].fill = header_fill
    ws_tcd[f'{col_letter}2'].font = header_font
    ws_tcd[f'{col_letter}2'].border = border
    ws_tcd[f'{col_letter}2'].alignment = center_align
    ws_tcd.column_dimensions[col_letter].width = 12

# --- EN-TÊTE LIGNES (Genres depuis CALC!A) - DYNAMIQUES ---
for row_idx in range(1, len_dict['len_gender']):
    row_num = 2 + row_idx
    
    # N'affiche le genre QUE si :
    # - Le filtre Genre = "Tous" (affiche tous les genres)
    # - OU le genre de la ligne = le filtre Genre
    ws_tcd[f'A{row_num}'] = (
        f'=IF(OR(TDB1!$G$3="Tous", IFERROR(INDEX(CALC!$A$2:$A$100,{row_idx}),)=TDB1!$G$3), '
        f'IFERROR(INDEX(CALC!$A$2:$A$100,{row_idx}),""), "")'
    )
    ws_tcd[f'A{row_num}'].fill = row_header_fill
    ws_tcd[f'A{row_num}'].font = row_header_font
    ws_tcd[f'A{row_num}'].border = border
    ws_tcd[f'A{row_num}'].alignment = center_align

# --- DONNÉES : COUNTIFS avec affichage conditionnel ---
for row_idx in range(1, len_dict['len_gender']):
    row_num = 2 + row_idx
    for col_idx in range(1, len_dict['len_age']):
        col_letter = get_column_letter(col_idx + 1)
        
        # Affiche la valeur SEULEMENT si :
        # - L'âge de la colonne est affiché (correspond au filtre)
        # - ET le genre de la ligne est affiché (correspond au filtre)
        formula = (
            f'=IF(AND('
            f'OR(TDB1!$D$3="Tous", IFERROR(INDEX(CALC!$C$2:$C$100,{col_idx}),)=TDB1!$D$3), '
            f'OR(TDB1!$G$3="Tous", IFERROR(INDEX(CALC!$A$2:$A$100,{row_idx}),)=TDB1!$G$3)'
            f'), '
            f'IFERROR(COUNTIFS('
            f'DATA!$M:$M, 1, '
            f'DATA!$B:$B, IF(TDB1!$G$3="Tous", IFERROR(INDEX(CALC!$A$2:$A$100,{row_idx}),""), TDB1!$G$3), '
            f'DATA!$A:$A, IF(TDB1!$D$3="Tous", IFERROR(INDEX(CALC!$C$2:$C$100,{col_idx}),), TDB1!$D$3), '
            f'DATA!$D:$D, IF(TDB1!$J$3="Tous", "<>", TDB1!$J$3), '
            f'DATA!$I:$I, IF(TDB1!$M$3="Tous", "<>", TDB1!$M$3)'
            f'), 0), "")'
        )
        
        ws_tcd[f'{col_letter}{row_num}'] = formula
        ws_tcd[f'{col_letter}{row_num}'].border = border
        ws_tcd[f'{col_letter}{row_num}'].alignment = center_align
        ws_tcd[f'{col_letter}{row_num}'].number_format = '0'

print("\n✅ Tableau TCD 'Dépression par Genre et Âge' créé (DYNAMIQUE) !")
print(f"   📊 Comportement :")
print(f"      ✓ Filtre Âge='Tous' + Genre='Tous' → Tableau complet (tous les genres × tous les âges)")
print(f"      ✓ Filtre Âge='19' + Genre='Tous' → 1 colonne (19) × tous les genres")
print(f"      ✓ Filtre Âge='Tous' + Genre='Female' → toutes les colonnes × 1 ligne (Female)")
print(f"      ✓ Filtre Âge='19' + Genre='Female' → 1 colonne (19) × 1 ligne (Female)\n")

wb.save(path_file)
wb.close()

print(f"📋 Feuilles présentes : {wb.sheetnames}\n")


✅ Tableau TCD 'Dépression par Genre et Âge' créé (DYNAMIQUE) !
   📊 Comportement :
      ✓ Filtre Âge='Tous' + Genre='Tous' → Tableau complet (tous les genres × tous les âges)
      ✓ Filtre Âge='19' + Genre='Tous' → 1 colonne (19) × tous les genres
      ✓ Filtre Âge='Tous' + Genre='Female' → toutes les colonnes × 1 ligne (Female)
      ✓ Filtre Âge='19' + Genre='Female' → 1 colonne (19) × 1 ligne (Female)

📋 Feuilles présentes : ['TDB1', 'DATA', 'Correlations', 'CALC', 'TCD']



### Tableau Addiction par genre et âge

In [1685]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

wb = load_workbook(path_file)

# ========== STYLES ==========
title_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")  # Bleu foncé
title_font = Font(bold=True, size=12, color="FFFFFF")
header_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")  # Teal foncé
header_font = Font(color="FFFFFF", bold=True, size=10)
row_header_fill = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")  # Teal clair
row_header_font = Font(bold=True, color="000000", size=10)
border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)
center_align = Alignment(horizontal="center", vertical="center")

# ========== CRÉATION DE LA FEUILLE ==========
if "TCD" in wb.sheetnames:
    ws_tcd = wb["TCD"]
else:
    ws_tcd = wb.create_sheet("TCD")

# =================================================================
# TABLEAU 2 : Addiction Moyenne par Genre et Âge (DYNAMIQUE)
# =================================================================

# --- TITRE (commencer à la ligne 10) ---
start_row = 10
ws_tcd[f'A{start_row}'] = "Addiction Moyenne par Genre et Âge"
ws_tcd[f'A{start_row}'].font = title_font
ws_tcd[f'A{start_row}'].fill = title_fill
ws_tcd[f'A{start_row}'].alignment = center_align
ws_tcd.merge_cells(f'A{start_row}:J{start_row}')
ws_tcd.row_dimensions[start_row].height = 25

# --- EN-TÊTE COLONNES (Âges depuis CALC!C) - DYNAMIQUES ---
header_row = start_row + 1
ws_tcd[f'A{header_row}'] = "Genre"
ws_tcd[f'A{header_row}'].fill = header_fill
ws_tcd[f'A{header_row}'].font = header_font
ws_tcd[f'A{header_row}'].border = border
ws_tcd[f'A{header_row}'].alignment = center_align
ws_tcd.column_dimensions['A'].width = 15

# Insérer tous les âges en en-tête de colonne
for col_idx in range(1, len_dict['len_age']):
    col_letter = get_column_letter(col_idx + 1)
    
    # N'affiche l'âge QUE si :
    # - Le filtre Âge = "Tous" (affiche tous les âges)
    # - OU l'âge de la colonne = le filtre Âge
    ws_tcd[f'{col_letter}{header_row}'] = (
        f'=IF(OR(TDB1!$D$3="Tous", IFERROR(INDEX(CALC!$C$2:$C$100,{col_idx}),)=TDB1!$D$3), '
        f'IFERROR(INDEX(CALC!$C$2:$C$100,{col_idx}),""), "")'
    )
    ws_tcd[f'{col_letter}{header_row}'].fill = header_fill
    ws_tcd[f'{col_letter}{header_row}'].font = header_font
    ws_tcd[f'{col_letter}{header_row}'].border = border
    ws_tcd[f'{col_letter}{header_row}'].alignment = center_align
    ws_tcd.column_dimensions[col_letter].width = 12

# --- EN-TÊTE LIGNES (Genres depuis CALC!A) - DYNAMIQUES ---
for row_idx in range(1, len_dict['len_gender']):
    row_num = header_row + row_idx
    
    # N'affiche le genre QUE si :
    # - Le filtre Genre = "Tous" (affiche tous les genres)
    # - OU le genre de la ligne = le filtre Genre
    ws_tcd[f'A{row_num}'] = (
        f'=IF(OR(TDB1!$G$3="Tous", IFERROR(INDEX(CALC!$A$2:$A$100,{row_idx}),)=TDB1!$G$3), '
        f'IFERROR(INDEX(CALC!$A$2:$A$100,{row_idx}),""), "")'
    )
    ws_tcd[f'A{row_num}'].fill = row_header_fill
    ws_tcd[f'A{row_num}'].font = row_header_font
    ws_tcd[f'A{row_num}'].border = border
    ws_tcd[f'A{row_num}'].alignment = center_align

# --- DONNÉES : AVERAGEIFS avec affichage conditionnel ---
for row_idx in range(1, len_dict['len_gender']):
    row_num = header_row + row_idx
    for col_idx in range(1, len_dict['len_age']):
        col_letter = get_column_letter(col_idx + 1)
        
        # Affiche la MOYENNE SEULEMENT si :
        # - L'âge de la colonne est affiché (correspond au filtre)
        # - ET le genre de la ligne est affiché (correspond au filtre)
        formula = (
            f'=IF(AND('
            f'OR(TDB1!$D$3="Tous", IFERROR(INDEX(CALC!$C$2:$C$100,{col_idx}),)=TDB1!$D$3), '
            f'OR(TDB1!$G$3="Tous", IFERROR(INDEX(CALC!$A$2:$A$100,{row_idx}),)=TDB1!$G$3)'
            f'), '
            f'IFERROR(AVERAGEIFS('
            f'DATA!$L:$L, '  # Colonne addiction_level
            f'DATA!$B:$B, IF(TDB1!$G$3="Tous", IFERROR(INDEX(CALC!$A$2:$A$100,{row_idx}),""), TDB1!$G$3), '
            f'DATA!$A:$A, IF(TDB1!$D$3="Tous", IFERROR(INDEX(CALC!$C$2:$C$100,{col_idx}),), TDB1!$D$3), '
            f'DATA!$D:$D, IF(TDB1!$J$3="Tous", "<>", TDB1!$J$3), '
            f'DATA!$I:$I, IF(TDB1!$M$3="Tous", "<>", TDB1!$M$3)'
            f'), 0), "")'
        )
        
        ws_tcd[f'{col_letter}{row_num}'] = formula
        ws_tcd[f'{col_letter}{row_num}'].border = border
        ws_tcd[f'{col_letter}{row_num}'].alignment = center_align
        ws_tcd[f'{col_letter}{row_num}'].number_format = '0.00'  # Format décimal



wb.save(path_file)
wb.close()

print(f"📋 Feuilles présentes : {wb.sheetnames}\n")

📋 Feuilles présentes : ['TDB1', 'DATA', 'Correlations', 'CALC', 'TCD']



### Summary pour boite à moustache

### KPI

In [1686]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.worksheet.formula import ArrayFormula

wb = load_workbook(path_file)

# ========== STYLES ==========
title_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")
title_font = Font(bold=True, size=12, color="FFFFFF")
header_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True, size=10)
row_header_fill = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
row_header_font = Font(bold=True, color="000000", size=10)
border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)
center_align = Alignment(horizontal="center", vertical="center")

if "TCD" in wb.sheetnames:
    ws_tcd = wb["TCD"]
else:
    ws_tcd = wb.create_sheet("TCD")

# =================================================================
# DÉFINITION DES FILTRES
# =================================================================

# Filter complet : Genre + Âge + Plateforme + Interaction
filter_rng_complet = (
    ', DATA!$B$2:$B$1201, IF(TDB1!$G$3="Tous", "<>", TDB1!$G$3)'
    ', DATA!$A$2:$A$1201, IF(TDB1!$D$3="Tous", "<>", TDB1!$D$3)'
    ', DATA!$D$2:$D$1201, IF(TDB1!$J$3="Tous", "<>", TDB1!$J$3)'
    ', DATA!$I$2:$I$1201, IF(TDB1!$M$3="Tous", "<>", TDB1!$M$3)'
)

# Filter SANS Plateforme (pour trouver le réseau le plus utilisé)
filter_rng_sans_plateforme = (
    ', DATA!$B$2:$B$1201, IF(TDB1!$G$3="Tous", "<>", TDB1!$G$3)'
    ', DATA!$A$2:$A$1201, IF(TDB1!$D$3="Tous", "<>", TDB1!$D$3)'
    ', DATA!$I$2:$I$1201, IF(TDB1!$M$3="Tous", "<>", TDB1!$M$3)'
)

# Filtres matriciels pour ArrayFormula
filter_arr = (
    ' * IF(TDB1!$G$3="Tous", 1, DATA!$B$2:$B$1201=TDB1!$G$3)'
    ' * IF(TDB1!$D$3="Tous", 1, DATA!$A$2:$A$1201=TDB1!$D$3)'
    ' * IF(TDB1!$J$3="Tous", 1, DATA!$D$2:$D$1201=TDB1!$J$3)'
    ' * IF(TDB1!$M$3="Tous", 1, DATA!$I$2:$I$1201=TDB1!$M$3)'
)

# Filtres matriciels SANS Plateforme
filter_arr_sans_plateforme = (
    ' * IF(TDB1!$G$3="Tous", 1, DATA!$B$2:$B$1201=TDB1!$G$3)'
    ' * IF(TDB1!$D$3="Tous", 1, DATA!$A$2:$A$1201=TDB1!$D$3)'
    ' * IF(TDB1!$M$3="Tous", 1, DATA!$I$2:$I$1201=TDB1!$M$3)'
)

# =================================================================
# TABLEAU 3 : Summary Statistiques Addiction par Âge
# =================================================================

start_row = 20

ws_tcd[f'A{start_row}'] = "Statistiques Addiction par Âge (pour Boxplot)"
ws_tcd[f'A{start_row}'].font = title_font
ws_tcd[f'A{start_row}'].fill = title_fill
ws_tcd[f'A{start_row}'].alignment = center_align
ws_tcd.merge_cells(f'A{start_row}:H{start_row}')
ws_tcd.row_dimensions[start_row].height = 25

colonnes = ["Âge", "Min", "Q1", "Médiane", "Q3", "Max", "Moyenne", "Écart-type"]
for col_idx, col_name in enumerate(colonnes):
    col_letter = get_column_letter(col_idx + 1)
    ws_tcd[f'{col_letter}{start_row+1}'] = col_name
    ws_tcd[f'{col_letter}{start_row+1}'].fill = header_fill
    ws_tcd[f'{col_letter}{start_row+1}'].font = header_font
    ws_tcd[f'{col_letter}{start_row+1}'].border = border
    ws_tcd[f'{col_letter}{start_row+1}'].alignment = center_align
    ws_tcd.column_dimensions[col_letter].width = 12

num_ages = len_dict['len_age']

for row_idx in range(1, num_ages):
    row_num = start_row + 1 + row_idx
    
    ws_tcd[f'A{row_num}'] = f'=IFERROR(INDEX(CALC!$C$2:$C$1000,{row_idx}),"")'
    ws_tcd[f'A{row_num}'].fill = row_header_fill
    ws_tcd[f'A{row_num}'].font = row_header_font
    ws_tcd[f'A{row_num}'].border = border
    ws_tcd[f'A{row_num}'].alignment = center_align
    
    ws_tcd[f'B{row_num}'] = ArrayFormula(
        f'B{row_num}', 
        f'=MIN(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201))'
    )
    
    ws_tcd[f'C{row_num}'] = ArrayFormula(
        f'C{row_num}', 
        f'=QUARTILE(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201), 1)'
    )
    
    ws_tcd[f'D{row_num}'] = ArrayFormula(
        f'D{row_num}', 
        f'=MEDIAN(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201))'
    )
    
    ws_tcd[f'E{row_num}'] = ArrayFormula(
        f'E{row_num}', 
        f'=QUARTILE(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201), 3)'
    )
    
    ws_tcd[f'F{row_num}'] = ArrayFormula(
        f'F{row_num}', 
        f'=MAX(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201))'
    )
    
    ws_tcd[f'G{row_num}'] = f'=IFERROR(AVERAGEIFS(DATA!$L$2:$L$1201, DATA!$A$2:$A$1201, $A{row_num}, DATA!$M$2:$M$1201, 1{filter_rng_complet}), NA())'
    
    ws_tcd[f'H{row_num}'] = ArrayFormula(
        f'H{row_num}', 
        f'=STDEV(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201))'
    )
    
    for col_idx in range(2, 9):
        col_letter = get_column_letter(col_idx)
        cell = ws_tcd[f'{col_letter}{row_num}']
        cell.border = border
        cell.alignment = center_align
        cell.number_format = '0.00'



wb.save(path_file)
wb.close()

print(f"📋 Feuilles présentes : {wb.sheetnames}\n")

📋 Feuilles présentes : ['TDB1', 'DATA', 'Correlations', 'CALC', 'TCD']



In [1687]:
# from openpyxl import load_workbook
# from openpyxl.worksheet.formula import ArrayFormula

# wb = load_workbook(path_file)

# if "TCD" in wb.sheetnames:
#     ws_tcd = wb["TCD"]
# else:
#     ws_tcd = wb.create_sheet("TCD")

# # =================================================================
# # DÉFINITION DES FILTRES
# # =================================================================

# filter_rng = (
#     ', DATA!$B$2:$B$1201, IF(TDB1!$G$3="Tous", "<>", TDB1!$G$3)'
#     ', DATA!$A$2:$A$1201, IF(TDB1!$D$3="Tous", "<>", TDB1!$D$3)'
#     ', DATA!$D$2:$D$1201, IF(TDB1!$J$3="Tous", "<>", TDB1!$J$3)'
#     ', DATA!$I$2:$I$1201, IF(TDB1!$M$3="Tous", "<>", TDB1!$M$3)'
# )

# # =================================================================
# # KPI 1 : RÉSEAU LE PLUS UTILISÉ
# # =================================================================

# kpi_start_row = 30

# ws_tcd[f"A{kpi_start_row}"] = "Réseau le plus utilisé"

# # FORMULE ADAPTÉE : Compter les dépressifs (M=1) par plateforme avec filtres
# formule_plateforme = (
#     f'=INDEX(DATA!$D$2:$D$1201, '
#     f'MATCH(MAX(COUNTIFS(DATA!$D$2:$D$1201, DATA!$D$2:$D$1201, '
#     f'DATA!$M$2:$M$1201, 1{filter_rng})), '
#     f'COUNTIFS(DATA!$D$2:$D$1201, DATA!$D$2:$D$1201, '
#     f'DATA!$M$2:$M$1201, 1{filter_rng}), 0))'
# )

# ws_tcd[f"B{kpi_start_row}"] = ArrayFormula(f"B{kpi_start_row}", formule_plateforme)

# # =================================================================
# # KPI 2 : PERFORMANCE SCOLAIRE MOYENNE
# # =================================================================

# ws_tcd[f"A{kpi_start_row+1}"] = "Perf sco moyenne - dépressifs"

# formule_perf = (
#     f'=IFERROR(AVERAGEIFS(DATA!$G$2:$G$1201, '
#     f'DATA!$M$2:$M$1201, 1{filter_rng}), NA())'
# )

# ws_tcd[f"B{kpi_start_row+1}"] = formule_perf

# print("\n✅ KPI corrigés avec la formule adaptée !")
# print(f"   📊 KPI 1 : Réseau le plus utilisé")
# print(f"      ✓ Compte chaque plateforme chez les dépressifs")
# print(f"   📊 KPI 2 : Performance scolaire moyenne")
# print(f"   🔗 Filtres appliqués : Genre (G3) + Âge (D3) + Plateforme (J3) + Interaction (M3)\n")

# wb.save(path_file)
# wb.close()

# print(f"📋 Feuilles présentes : {wb.sheetnames}\n")

In [1688]:
from openpyxl import load_workbook
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

wb = load_workbook(path_file)

if "TCD" in wb.sheetnames:
    ws_tcd = wb["TCD"]
else:
    ws_tcd = wb.create_sheet("TCD")

# ========== STYLES ==========
kpi_title_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")
kpi_title_font = Font(bold=True, color="FFFFFF", size=11)
kpi_value_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")
kpi_value_font = Font(bold=True, size=16, color="FFFFFF")
kpi_label_font = Font(bold=True, size=10, color="000000")
border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)
center_align = Alignment(horizontal="center", vertical="center", wrap_text=True)

# =================================================================
# DÉFINITION DES FILTRES
# =================================================================

filter_rng = (
    ', DATA!$B$2:$B$1201, IF(TDB1!$G$3="Tous", "<>", TDB1!$G$3)'
    ', DATA!$A$2:$A$1201, IF(TDB1!$D$3="Tous", "<>", TDB1!$D$3)'
    ', DATA!$D$2:$D$1201, IF(TDB1!$J$3="Tous", "<>", TDB1!$J$3)'
    ', DATA!$I$2:$I$1201, IF(TDB1!$M$3="Tous", "<>", TDB1!$M$3)'
)

# Filtres matriciels pour ArrayFormula
filter_arr = (
    ' * IF(TDB1!$G$3="Tous", 1, DATA!$B$2:$B$1201=TDB1!$G$3)'
    ' * IF(TDB1!$D$3="Tous", 1, DATA!$A$2:$A$1201=TDB1!$D$3)'
    ' * IF(TDB1!$J$3="Tous", 1, DATA!$D$2:$D$1201=TDB1!$J$3)'
    ' * IF(TDB1!$M$3="Tous", 1, DATA!$I$2:$I$1201=TDB1!$M$3)'
)

# =================================================================
# KPI SECTION
# =================================================================

kpi_start_row = 30

# --- TITRE SECTION KPI ---
ws_tcd[f'A{kpi_start_row}'] = "Indicateurs Clés (KPI) - Personnes Dépressives"
ws_tcd[f'A{kpi_start_row}'].font = Font(bold=True, size=12, color="FFFFFF")
ws_tcd[f'A{kpi_start_row}'].fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")
ws_tcd[f'A{kpi_start_row}'].alignment = center_align
ws_tcd.merge_cells(f'A{kpi_start_row}:H{kpi_start_row}')
ws_tcd.row_dimensions[kpi_start_row].height = 25

# ========== KPI 1 : RÉSEAU LE PLUS UTILISÉ ==========
kpi1_row = kpi_start_row + 2

ws_tcd[f'A{kpi1_row}'] = "Réseau le plus utilisé"
ws_tcd[f'A{kpi1_row}'].fill = kpi_title_fill
ws_tcd[f'A{kpi1_row}'].font = kpi_title_font
ws_tcd[f'A{kpi1_row}'].border = border
ws_tcd[f'A{kpi1_row}'].alignment = center_align
ws_tcd.merge_cells(f'A{kpi1_row}:C{kpi1_row}')
ws_tcd.row_dimensions[kpi1_row].height = 20

kpi1_value_row = kpi1_row + 1
formule_plateforme = (
    f'=INDEX(DATA!$D$2:$D$1201, '
    f'MATCH(MAX(COUNTIFS(DATA!$D$2:$D$1201, DATA!$D$2:$D$1201, '
    f'DATA!$M$2:$M$1201, 1{filter_rng})), '
    f'COUNTIFS(DATA!$D$2:$D$1201, DATA!$D$2:$D$1201, '
    f'DATA!$M$2:$M$1201, 1{filter_rng}), 0))'
)
ws_tcd[f'A{kpi1_value_row}'] = ArrayFormula(f"A{kpi1_value_row}", formule_plateforme)
ws_tcd[f'A{kpi1_value_row}'].fill = kpi_value_fill
ws_tcd[f'A{kpi1_value_row}'].font = kpi_value_font
ws_tcd[f'A{kpi1_value_row}'].border = border
ws_tcd[f'A{kpi1_value_row}'].alignment = center_align
ws_tcd.merge_cells(f'A{kpi1_value_row}:C{kpi1_value_row}')
ws_tcd.row_dimensions[kpi1_value_row].height = 35

# Nombre de dépressifs
kpi1_count_row = kpi1_value_row + 1
ws_tcd[f'A{kpi1_count_row}'] = "Nombre de dépressifs"
ws_tcd[f'A{kpi1_count_row}'].font = kpi_label_font
ws_tcd[f'A{kpi1_count_row}'].border = border
ws_tcd[f'A{kpi1_count_row}'].alignment = Alignment(horizontal="left", vertical="center")

ws_tcd[f'B{kpi1_count_row}'] = (
    f'=IFERROR(COUNTIFS(DATA!$M$2:$M$1201, 1{filter_rng}), 0)'
)
ws_tcd[f'B{kpi1_count_row}'].font = Font(bold=True, size=11)
ws_tcd[f'B{kpi1_count_row}'].border = border
ws_tcd[f'B{kpi1_count_row}'].alignment = center_align
ws_tcd[f'B{kpi1_count_row}'].number_format = '0'

ws_tcd.column_dimensions['A'].width = 18
ws_tcd.column_dimensions['B'].width = 12
ws_tcd.column_dimensions['C'].width = 12

# ========== KPI 2 : PERFORMANCE SCOLAIRE MOYENNE ==========
kpi2_row = kpi_start_row + 2

ws_tcd[f'D{kpi2_row}'] = "Performance scolaire moyenne"
ws_tcd[f'D{kpi2_row}'].fill = kpi_title_fill
ws_tcd[f'D{kpi2_row}'].font = kpi_title_font
ws_tcd[f'D{kpi2_row}'].border = border
ws_tcd[f'D{kpi2_row}'].alignment = center_align
ws_tcd.merge_cells(f'D{kpi2_row}:F{kpi2_row}')
ws_tcd.row_dimensions[kpi2_row].height = 20

kpi2_value_row = kpi2_row + 1
ws_tcd[f'D{kpi2_value_row}'] = (
    f'=IFERROR(AVERAGEIFS(DATA!$G$2:$G$1201, '
    f'DATA!$M$2:$M$1201, 1{filter_rng}), NA())'
)
ws_tcd[f'D{kpi2_value_row}'].fill = kpi_value_fill
ws_tcd[f'D{kpi2_value_row}'].font = kpi_value_font
ws_tcd[f'D{kpi2_value_row}'].border = border
ws_tcd[f'D{kpi2_value_row}'].alignment = center_align
ws_tcd.merge_cells(f'D{kpi2_value_row}:F{kpi2_value_row}')
ws_tcd.row_dimensions[kpi2_value_row].height = 35
ws_tcd[f'D{kpi2_value_row}'].number_format = '0.00'

# Min / Max Performance - CORRECTION : Utiliser ArrayFormula
kpi2_minmax_row = kpi2_value_row + 1
ws_tcd[f'D{kpi2_minmax_row}'] = "Min / Max"
ws_tcd[f'D{kpi2_minmax_row}'].font = kpi_label_font
ws_tcd[f'D{kpi2_minmax_row}'].border = border
ws_tcd[f'D{kpi2_minmax_row}'].alignment = Alignment(horizontal="left", vertical="center")

# MIN avec ArrayFormula
ws_tcd[f'E{kpi2_minmax_row}'] = ArrayFormula(
    f'E{kpi2_minmax_row}',
    f'=MIN(IF((DATA!$M$2:$M$1201=1){filter_arr}, DATA!$G$2:$G$1201))'
)
ws_tcd[f'E{kpi2_minmax_row}'].font = Font(size=9, bold=True)
ws_tcd[f'E{kpi2_minmax_row}'].border = border
ws_tcd[f'E{kpi2_minmax_row}'].alignment = center_align
ws_tcd[f'E{kpi2_minmax_row}'].number_format = '0.00'

# MAX avec ArrayFormula
ws_tcd[f'F{kpi2_minmax_row}'] = ArrayFormula(
    f'F{kpi2_minmax_row}',
    f'=MAX(IF((DATA!$M$2:$M$1201=1){filter_arr}, DATA!$G$2:$G$1201))'
)
ws_tcd[f'F{kpi2_minmax_row}'].font = Font(size=9, bold=True)
ws_tcd[f'F{kpi2_minmax_row}'].border = border
ws_tcd[f'F{kpi2_minmax_row}'].alignment = center_align
ws_tcd[f'F{kpi2_minmax_row}'].number_format = '0.00'

ws_tcd.column_dimensions['D'].width = 18
ws_tcd.column_dimensions['E'].width = 12
ws_tcd.column_dimensions['F'].width = 12

print("\n✅ KPI avec Min/Max corrigés (ArrayFormula) !")
print(f"   📊 KPI 1 - Réseau le plus utilisé")
print(f"      ✓ Plateforme la plus utilisée")
print(f"      ✓ Nombre total de dépressifs")
print(f"   📊 KPI 2 - Performance scolaire")
print(f"      ✓ Moyenne")
print(f"      ✓ Min / Max (avec ArrayFormula)\n")

# # 1. Création du Tableau Tampon (ligne 100) dans TCD
# # On utilise COUNTIFS qui s'ajuste dynamiquement aux filtres TDB1
# # Ce tableau est la source de vérité pour le PieChart
# ws_tcd[f'A100'] = "Genre"
# ws_tcd[f'B100'] = "Total Dépressifs"

# # Ligne pour Male
# ws_tcd['A101'] = "male"
# ws_tcd['B101'] = '=COUNTIFS(DATA!$M:$M, 1, DATA!$B:$B, "male", DATA!$D:$D, IF(TDB1!$J$3="Tous", "*", TDB1!$J$3), DATA!$I:$I, IF(TDB1!$M$3="Tous", "*", TDB1!$M$3), DATA!$A:$A, IF(TDB1!$D$3="Tous", "*", TDB1!$D$3))'

# # Ligne pour Female
# ws_tcd['A102'] = "female"
# ws_tcd['B102'] = '=COUNTIFS(DATA!$M:$M, 1, DATA!$B:$B, "female", DATA!$D:$D, IF(TDB1!$J$3="Tous", "*", TDB1!$J$3), DATA!$I:$I, IF(TDB1!$M$3="Tous", "*", TDB1!$M$3), DATA!$A:$A, IF(TDB1!$D$3="Tous", "*", TDB1!$D$3))'

wb.save(path_file)
wb.close()

print(f"📋 Feuilles présentes : {wb.sheetnames}\n")


✅ KPI avec Min/Max corrigés (ArrayFormula) !
   📊 KPI 1 - Réseau le plus utilisé
      ✓ Plateforme la plus utilisée
      ✓ Nombre total de dépressifs
   📊 KPI 2 - Performance scolaire
      ✓ Moyenne
      ✓ Min / Max (avec ArrayFormula)

📋 Feuilles présentes : ['TDB1', 'DATA', 'Correlations', 'CALC', 'TCD']



# Création des graphiques

### BarChart - Niveau d'addiction moyen selon l'âge et le genre

In [1689]:
from openpyxl import load_workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.chart.label import DataLabelList

wb = load_workbook(path_file)

ws_tcd = wb["TCD"]
ws_tdb1 = wb["TDB1"]

# =================================================================
# GRAPHIQUE 1 : Addiction Moyenne par Âge et Genre
# =================================================================

chart_addiction = BarChart()
chart_addiction.type = "col"
chart_addiction.style = 10
chart_addiction.title = "Addiction moyenne par âge et genre"
chart_addiction.y_axis.title = "Niveau d'addiction"
chart_addiction.x_axis.title = "Âge"
chart_addiction.height = 5
chart_addiction.width = 15

# Références de données
max_col = len_dict['len_age']
data = Reference(ws_tcd, min_col=1, min_row=12, max_col=max_col, max_row=13)
cats = Reference(ws_tcd, min_col=2, min_row=11, max_col=max_col, max_row=11)

chart_addiction.add_data(data, titles_from_data=True, from_rows=True)
chart_addiction.set_categories(cats)
chart_addiction.overlap = -15

for row in ws_tcd.iter_rows(min_row=12, max_row=13, min_col=1, max_col=max_col):
    for cell in row:
        cell.number_format = '0'

# Configuration simple des étiquettes (valeur affichée, position externe)
for series in chart_addiction.series:
    series.dLbls = DataLabelList()
    series.dLbls.showVal = True
    series.dLbls.showCatName = False
    series.dLbls.showSerName = False
    series.dLbls.position = 'outEnd'
    series.dLbls.numFmt = "0"

# Application des couleurs aux barres
color_male = "040459"
color_female = "7d093f"

if len(chart_addiction.series) >= 2:
    # Série 0 (Male)
    chart_addiction.series[0].graphicalProperties.solidFill = color_male
    for point in chart_addiction.series[0].data_points:
        point.graphicalProperties.solidFill = color_male
    
    # Série 1 (Female)
    chart_addiction.series[1].graphicalProperties.solidFill = color_female
    for point in chart_addiction.series[1].data_points:
        point.graphicalProperties.solidFill = color_female

chart_addiction.legend.position = "r"
chart_addiction.y_axis.majorGridlines = None

ws_tdb1.add_chart(chart_addiction, "B5")

wb.save(path_file)
wb.close()

print("✅ Graphique créé avec succès (couleurs barres appliquées, étiquettes activées).")

✅ Graphique créé avec succès (couleurs barres appliquées, étiquettes activées).


### PieChart - Répartition de la dépression selon le genre

In [1690]:
from openpyxl import load_workbook
from openpyxl.chart import PieChart, Reference
from openpyxl.chart.label import DataLabelList
from openpyxl.chart.series import DataPoint
from openpyxl.styles import Font

wb = load_workbook(path_file)

ws_tcd = wb["TCD"]
ws_tdb1 = wb["TDB1"]

# =================================================================
# HELPER : Dépression par Genre - Pour TDB1 (ligne 120)
# =================================================================

start_row_pie = 120

ws_tcd[f"A{start_row_pie}"] = "Total Dépression Genre"
ws_tcd[f"A{start_row_pie}"].font = Font(bold=True, size=9)

ws_tcd[f"A{start_row_pie+1}"] = "Genre"
ws_tcd[f"B{start_row_pie+1}"] = "Total"

# Male - Avec condition IF pour filtrer par genre
ws_tcd[f"A{start_row_pie+2}"] = "Male"
ws_tcd[f"B{start_row_pie+2}"] = (
    f'=IF(OR(TDB1!$G$3="Tous", TDB1!$G$3="Male"), '
    f'IFERROR(COUNTIFS(DATA!$M$2:$M$1201, 1, DATA!$B$2:$B$1201, "male", '
    f'DATA!$D$2:$D$1201, IF(TDB1!$J$3="Tous", "<>", TDB1!$J$3), '
    f'DATA!$I$2:$I$1201, IF(TDB1!$M$3="Tous", "<>", TDB1!$M$3), '
    f'DATA!$A$2:$A$1201, IF(TDB1!$D$3="Tous", "<>", TDB1!$D$3)), 0), 0)'
)
ws_tcd[f"B{start_row_pie+2}"].number_format = '0'

# Female - Avec condition IF pour filtrer par genre
ws_tcd[f"A{start_row_pie+3}"] = "Female"
ws_tcd[f"B{start_row_pie+3}"] = (
    f'=IF(OR(TDB1!$G$3="Tous", TDB1!$G$3="Female"), '
    f'IFERROR(COUNTIFS(DATA!$M$2:$M$1201, 1, DATA!$B$2:$B$1201, "female", '
    f'DATA!$D$2:$D$1201, IF(TDB1!$J$3="Tous", "<>", TDB1!$J$3), '
    f'DATA!$I$2:$I$1201, IF(TDB1!$M$3="Tous", "<>", TDB1!$M$3), '
    f'DATA!$A$2:$A$1201, IF(TDB1!$D$3="Tous", "<>", TDB1!$D$3)), 0), 0)'
)
ws_tcd[f"B{start_row_pie+3}"].number_format = '0'

# =================================================================
# PIECHART : Répartition de la Dépression (TDB1)
# =================================================================

pie_chart = PieChart()
pie_chart.title = "Répartition de la dépression selon le genre"
pie_chart.height = 5
pie_chart.width = 10

# Données du graphique
data_pie = Reference(
    ws_tcd,
    min_col=2,
    min_row=start_row_pie + 1,
    max_row=start_row_pie + 3
)

cats_pie = Reference(
    ws_tcd,
    min_col=1,
    min_row=start_row_pie + 2,
    max_row=start_row_pie + 3
)

pie_chart.add_data(data_pie, titles_from_data=True)
pie_chart.set_categories(cats_pie)

# Affichage
pie_chart.dLbls = DataLabelList()
pie_chart.dLbls.showPercent = True
pie_chart.dLbls.showVal = False
pie_chart.dLbls.showCatName = True
pie_chart.legend = None

# Couleurs
colors = ["040459", "7D093F"]
series = pie_chart.series[0]

for idx, color in enumerate(colors):
    pt = DataPoint(idx=idx)
    pt.graphicalProperties.solidFill = color
    series.dPt.append(pt)

# Placement sur TDB1
ws_tdb1.add_chart(pie_chart, "J5")

wb.save(path_file)
wb.close()

print(f"✅ Fichier sauvegardé !\n")

✅ Fichier sauvegardé !



### Boîte à moustache - Répartition niveau addiction selon l'âge

In [1691]:
from openpyxl import load_workbook
from openpyxl.chart import LineChart, Reference, BarChart
from openpyxl.chart.label import DataLabelList

wb = load_workbook(path_file)

ws_tcd = wb["TCD"]
ws_tdb1 = wb["TDB1"]

# =================================================================
# GRAPHIQUE 3 : BoxPlot - Répartition Niveau Addiction selon l'Âge
# =================================================================

summary_start_row = 21
num_ages = len_dict['len_age'] - 1

box_plot = BarChart()
box_plot.type = "col"
box_plot.style = 10
box_plot.title = "Répartition niveau Addiction selon l'âge"
box_plot.y_axis.title = "Niveau d'addiction"
box_plot.x_axis.title = "Âge"
box_plot.height = 7
box_plot.width = 15

# Catégories (Âges)
cats_box = Reference(
    ws_tcd,
    min_col=1,
    min_row=summary_start_row + 1,
    max_row=summary_start_row + num_ages
)

# Ajouter les séries de données
series_names = ["Min", "Q1", "Médiane", "Q3", "Max"]
colors = ["808080", "A6A6A6", "FF1493", "CCCCCC", "555555"]

for col_idx, (series_name, color) in enumerate(zip(series_names, colors)):
    col_letter = chr(66 + col_idx)
    
    data = Reference(
        ws_tcd,
        min_col=col_idx + 2,
        min_row=summary_start_row,
        max_row=summary_start_row + num_ages
    )
    
    box_plot.add_data(data, titles_from_data=True)
    
    if len(box_plot.series) > 0:
        box_plot.series[-1].graphicalProperties.solidFill = color

# Définir les catégories
box_plot.set_categories(cats_box)

# AJOUTER LES ÉTIQUETTES DE DONNÉES
for series in box_plot.series:
    series.dLbls = DataLabelList()
    series.dLbls.showVal = True        # Afficher les valeurs
    series.dLbls.showCatName = False
    series.dLbls.showSerName = False
    series.dLbls.position = 'outEnd'   # Position au-dessus des barres
    series.dLbls.numFmt = "0"        # Format avec 1 décimale

# Configuration du graphique
box_plot.overlap = -50
box_plot.legend.position = "r"
box_plot.y_axis.majorGridlines = None

# Placer le graphique
ws_tdb1.add_chart(box_plot, "B16")



wb.save(path_file)
wb.close()

print(f"✅ Fichier sauvegardé !\n")

✅ Fichier sauvegardé !



### KPI

In [1692]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

wb = load_workbook(path_file)

ws_tcd = wb["TCD"]
ws_tdb1 = wb["TDB1"]

# ========== STYLES ==========
kpi_title_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")
kpi_title_font = Font(bold=True, color="FFFFFF", size=10)
kpi_value_fill = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
kpi_value_font = Font(bold=True, size=14, color="FF003D6B")
kpi_label_font = Font(bold=True, size=9, color="000000")
border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)
center_align = Alignment(horizontal="center", vertical="center", wrap_text=True)

# =================================================================
# AFFICHAGE DES KPI DANS TDB1 - Position J16
# =================================================================

kpi_tdb_row = 16

# --- KPI 1 : Réseau le plus utilisé ---
ws_tdb1[f'J{kpi_tdb_row}'] = "Réseau populaire"
ws_tdb1[f'J{kpi_tdb_row}'].fill = kpi_title_fill
ws_tdb1[f'J{kpi_tdb_row}'].font = kpi_title_font
ws_tdb1[f'J{kpi_tdb_row}'].alignment = center_align
ws_tdb1[f'J{kpi_tdb_row}'].border = border
ws_tdb1.merge_cells(f'J{kpi_tdb_row}:K{kpi_tdb_row}')

# Valeur du réseau (référence TCD)
ws_tdb1[f'J{kpi_tdb_row + 1}'] = f'=TCD!A32'  # Plateforme la plus utilisée
ws_tdb1[f'J{kpi_tdb_row + 1}'].fill = kpi_value_fill
ws_tdb1[f'J{kpi_tdb_row + 1}'].font = kpi_value_font
ws_tdb1[f'J{kpi_tdb_row + 1}'].alignment = center_align
ws_tdb1[f'J{kpi_tdb_row + 1}'].border = border
ws_tdb1.merge_cells(f'J{kpi_tdb_row + 1}:K{kpi_tdb_row + 1}')
ws_tdb1.row_dimensions[kpi_tdb_row + 1].height = 30

# --- KPI 2 : Nombre de dépressifs ---
ws_tdb1[f'L{kpi_tdb_row}'] = "Dépressifs"
ws_tdb1[f'L{kpi_tdb_row}'].fill = kpi_title_fill
ws_tdb1[f'L{kpi_tdb_row}'].font = kpi_title_font
ws_tdb1[f'L{kpi_tdb_row}'].alignment = center_align
ws_tdb1[f'L{kpi_tdb_row}'].border = border
ws_tdb1.merge_cells(f'L{kpi_tdb_row}:M{kpi_tdb_row}')

# Valeur
ws_tdb1[f'L{kpi_tdb_row + 1}'] = f'=TCD!B34'  # Nombre de dépressifs
ws_tdb1[f'L{kpi_tdb_row + 1}'].fill = kpi_value_fill
ws_tdb1[f'L{kpi_tdb_row + 1}'].font = kpi_value_font
ws_tdb1[f'L{kpi_tdb_row + 1}'].alignment = center_align
ws_tdb1[f'L{kpi_tdb_row + 1}'].border = border
ws_tdb1.merge_cells(f'L{kpi_tdb_row + 1}:M{kpi_tdb_row + 1}')

# --- KPI 3 : Performance scolaire moyenne ---
ws_tdb1[f'J{kpi_tdb_row + 3}'] = "Performance scolaire"
ws_tdb1[f'J{kpi_tdb_row + 3}'].fill = kpi_title_fill
ws_tdb1[f'J{kpi_tdb_row + 3}'].font = kpi_title_font
ws_tdb1[f'J{kpi_tdb_row + 3}'].alignment = center_align
ws_tdb1[f'J{kpi_tdb_row + 3}'].border = border
ws_tdb1.merge_cells(f'J{kpi_tdb_row + 3}:K{kpi_tdb_row + 3}')

# Valeur moyenne
ws_tdb1[f'J{kpi_tdb_row + 4}'] = f'=TCD!D33'  # Moyenne
ws_tdb1[f'J{kpi_tdb_row + 4}'].fill = kpi_value_fill
ws_tdb1[f'J{kpi_tdb_row + 4}'].font = kpi_value_font
ws_tdb1[f'J{kpi_tdb_row + 4}'].alignment = center_align
ws_tdb1[f'J{kpi_tdb_row + 4}'].border = border
ws_tdb1[f'J{kpi_tdb_row + 4}'].number_format = '0.00'
ws_tdb1.merge_cells(f'J{kpi_tdb_row + 4}:K{kpi_tdb_row + 4}')
ws_tdb1.row_dimensions[kpi_tdb_row + 4].height = 30

# --- Min / Max Performance ---
ws_tdb1[f'L{kpi_tdb_row + 3}'] = "Min / Max"
ws_tdb1[f'L{kpi_tdb_row + 3}'].fill = kpi_title_fill
ws_tdb1[f'L{kpi_tdb_row + 3}'].font = kpi_title_font
ws_tdb1[f'L{kpi_tdb_row + 3}'].alignment = center_align
ws_tdb1[f'L{kpi_tdb_row + 3}'].border = border
ws_tdb1.merge_cells(f'L{kpi_tdb_row + 3}:M{kpi_tdb_row + 3}')

# Min
ws_tdb1[f'L{kpi_tdb_row + 4}'] = f'=TCD!E34'
ws_tdb1[f'L{kpi_tdb_row + 4}'].fill = kpi_value_fill
ws_tdb1[f'L{kpi_tdb_row + 4}'].font = Font(bold=True, size=11, color="FF003D6B")
ws_tdb1[f'L{kpi_tdb_row + 4}'].alignment = center_align
ws_tdb1[f'L{kpi_tdb_row + 4}'].border = border
ws_tdb1[f'L{kpi_tdb_row + 4}'].number_format = '0.00'

# Max
ws_tdb1[f'M{kpi_tdb_row + 4}'] = f'=TCD!F34'
ws_tdb1[f'M{kpi_tdb_row + 4}'].fill = kpi_value_fill
ws_tdb1[f'M{kpi_tdb_row + 4}'].font = Font(bold=True, size=11, color="FF003D6B")
ws_tdb1[f'M{kpi_tdb_row + 4}'].alignment = center_align
ws_tdb1[f'M{kpi_tdb_row + 4}'].border = border
ws_tdb1[f'M{kpi_tdb_row + 4}'].number_format = '0.00'

# Ajuster les largeurs des colonnes
ws_tdb1.column_dimensions['J'].width = 15
ws_tdb1.column_dimensions['K'].width = 12
ws_tdb1.column_dimensions['L'].width = 15
ws_tdb1.column_dimensions['M'].width = 12

print("\n✅ KPI affichés dans TDB1 !")
print(f"   📊 KPI 1 - Réseau populaire (J16:K17)")
print(f"   📊 KPI 2 - Nombre de dépressifs (L16:M17)")
print(f"   📊 KPI 3 - Performance scolaire (J19:K20)")
print(f"   📊 KPI 4 - Min / Max (L19:M20)")
print(f"   📌 Position : TDB1 à partir de J16\n")

wb.save(path_file)
wb.close()

print(f"✅ Fichier sauvegardé !\n")


✅ KPI affichés dans TDB1 !
   📊 KPI 1 - Réseau populaire (J16:K17)
   📊 KPI 2 - Nombre de dépressifs (L16:M17)
   📊 KPI 3 - Performance scolaire (J19:K20)
   📊 KPI 4 - Min / Max (L19:M20)
   📌 Position : TDB1 à partir de J16

✅ Fichier sauvegardé !



## Page 2 du Tableau de Bord

In [1693]:
wb = load_workbook(path_file)

# Nettoyer si existe déjà
if "TDB2" in wb.sheetnames:
    del wb["TDB2"]

TDB2 = wb.create_sheet("TDB2", 1)
TDB2.sheet_view.showGridLines = False

# ========== TITRE PRINCIPAL EN BLEU FONCÉ ==========
title_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type='solid')
title_font = Font(name='Calibri', size=14, bold=True, color='FFFFFF')
title_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

TDB2.merge_cells('A1:O1')
TDB2['A1'] = 'Impact des réseaux sociaux sur la santé mentale'
TDB2['A1'].fill = title_fill
TDB2['A1'].font = title_font
TDB2['A1'].alignment = title_alignment
TDB2.row_dimensions[1].height = 30

# ========== LIGNE DE FILTRES ==========
TDB2['A3'] = 'Filtres :'
TDB2['A3'].font = Font(size=10, bold=True)
TDB2['A3'].alignment = Alignment(horizontal='left', vertical='center')

# FILTRES HORIZONTAUX - IDENTIQUES À TDB1
add_filter(TDB2, 'C', 3, 'Âge', 'C', len_dict['len_age'], 'AE', 
           title_color='FF003D6B', value_color='FFFFFF')

add_filter(TDB2, 'F', 3, 'Genre', 'A', len_dict['len_gender'], 'AF',
           title_color='FF003D6B', value_color='FFFFFF')

add_filter(TDB2, 'I', 3, 'Plateforme', 'E', len_dict['len_platform_usage'], 'AG',
           title_color='FF003D6B', value_color='FFFFFF')

add_filter(TDB2, 'L', 3, 'Interaction', 'G', len_dict['len_social_interaction_level'], 'AH',
           title_color='FF003D6B', value_color='FFFFFF')

# Définir la hauteur de la ligne des filtres
TDB2.row_dimensions[3].height = 25

print("\n✅ TDB2 créée avec succès !")
print(f"   📊 Titre : 'Impact des réseaux sociaux sur la santé mentale'")
print(f"   🔧 Filtres : Âge, Genre, Plateforme, Interaction")
print(f"   📌 Position : Feuille 2\n")

wb.save(path_file)
print(f"📋 Feuilles présentes : {wb.sheetnames}\n")

wb.close()

print(f"✅ Fichier sauvegardé !\n")

✅ Filtre 'Âge' créé en C3:D3
✅ Filtre 'Genre' créé en F3:G3
✅ Filtre 'Plateforme' créé en I3:J3
✅ Filtre 'Interaction' créé en L3:M3

✅ TDB2 créée avec succès !
   📊 Titre : 'Impact des réseaux sociaux sur la santé mentale'
   🔧 Filtres : Âge, Genre, Plateforme, Interaction
   📌 Position : Feuille 2

📋 Feuilles présentes : ['TDB1', 'TDB2', 'DATA', 'Correlations', 'CALC', 'TCD']

✅ Fichier sauvegardé !



### KPI

In [1694]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

wb = load_workbook(path_file)

ws_tcd = wb["TCD"]
ws_tdb2 = wb["TDB2"]

# ========== STYLES ==========
kpi_title_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")
kpi_title_font = Font(bold=True, color="FFFFFF", size=11)
kpi_male_fill = PatternFill(start_color="040459", end_color="040459", fill_type="solid")
kpi_female_fill = PatternFill(start_color="7D093F", end_color="7D093F", fill_type="solid")
kpi_value_font = Font(bold=True, size=16, color="FFFFFF")
border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)
center_align = Alignment(horizontal="center", vertical="center")

# =================================================================
# TABLEAU HELPER 1 : Dépressifs par Genre avec filtre Genre
# =================================================================

helper_row_1 = 100

ws_tcd[f'A{helper_row_1}'] = "Genre"
ws_tcd[f'B{helper_row_1}'] = "Dépressifs"
ws_tcd[f'A{helper_row_1}'].font = Font(bold=True)
ws_tcd[f'B{helper_row_1}'].font = Font(bold=True)

# Male - Avec condition IF pour filtrer par genre
ws_tcd[f'A{helper_row_1 + 1}'] = "Male"
ws_tcd[f'B{helper_row_1 + 1}'] = (
    f'=IF(OR(TDB2!$G$3="Tous", TDB2!$G$3="Male"), '
    f'IFERROR(COUNTIFS(DATA!$M$2:$M$1201, 1, DATA!$B$2:$B$1201, "male", '
    f'DATA!$D$2:$D$1201, IF(TDB2!$J$3="Tous", "<>", TDB2!$J$3), '
    f'DATA!$I$2:$I$1201, IF(TDB2!$M$3="Tous", "<>", TDB2!$M$3), '
    f'DATA!$A$2:$A$1201, IF(TDB2!$D$3="Tous", "<>", TDB2!$D$3)), 0), 0)'
)
ws_tcd[f'B{helper_row_1 + 1}'].number_format = '0'

# Female - Avec condition IF pour filtrer par genre
ws_tcd[f'A{helper_row_1 + 2}'] = "Female"
ws_tcd[f'B{helper_row_1 + 2}'] = (
    f'=IF(OR(TDB2!$G$3="Tous", TDB2!$G$3="Female"), '
    f'IFERROR(COUNTIFS(DATA!$M$2:$M$1201, 1, DATA!$B$2:$B$1201, "female", '
    f'DATA!$D$2:$D$1201, IF(TDB2!$J$3="Tous", "<>", TDB2!$J$3), '
    f'DATA!$I$2:$I$1201, IF(TDB2!$M$3="Tous", "<>", TDB2!$M$3), '
    f'DATA!$A$2:$A$1201, IF(TDB2!$D$3="Tous", "<>", TDB2!$D$3)), 0), 0)'
)
ws_tcd[f'B{helper_row_1 + 2}'].number_format = '0'

# =================================================================
# TABLEAU HELPER 2 : Addiction Moyenne par Genre avec filtre Genre
# =================================================================

helper_row_2 = 105

ws_tcd[f'A{helper_row_2}'] = "Genre"
ws_tcd[f'B{helper_row_2}'] = "Addiction Moyenne"
ws_tcd[f'A{helper_row_2}'].font = Font(bold=True)
ws_tcd[f'B{helper_row_2}'].font = Font(bold=True)

# Male - Avec condition IF pour filtrer par genre
ws_tcd[f'A{helper_row_2 + 1}'] = "Male"
ws_tcd[f'B{helper_row_2 + 1}'] = (
    f'=IF(OR(TDB2!$G$3="Tous", TDB2!$G$3="Male"), '
    f'IFERROR(AVERAGEIFS(DATA!$L$2:$L$1201, DATA!$B$2:$B$1201, "male", '
    f'DATA!$D$2:$D$1201, IF(TDB2!$J$3="Tous", "<>", TDB2!$J$3), '
    f'DATA!$I$2:$I$1201, IF(TDB2!$M$3="Tous", "<>", TDB2!$M$3), '
    f'DATA!$A$2:$A$1201, IF(TDB2!$D$3="Tous", "<>", TDB2!$D$3)), 0), 0)'
)
ws_tcd[f'B{helper_row_2 + 1}'].number_format = '0.0'

# Female - Avec condition IF pour filtrer par genre
ws_tcd[f'A{helper_row_2 + 2}'] = "Female"
ws_tcd[f'B{helper_row_2 + 2}'] = (
    f'=IF(OR(TDB2!$G$3="Tous", TDB2!$G$3="Female"), '
    f'IFERROR(AVERAGEIFS(DATA!$L$2:$L$1201, DATA!$B$2:$B$1201, "female", '
    f'DATA!$D$2:$D$1201, IF(TDB2!$J$3="Tous", "<>", TDB2!$J$3), '
    f'DATA!$I$2:$I$1201, IF(TDB2!$M$3="Tous", "<>", TDB2!$M$3), '
    f'DATA!$A$2:$A$1201, IF(TDB2!$D$3="Tous", "<>", TDB2!$D$3)), 0), 0)'
)
ws_tcd[f'B{helper_row_2 + 2}'].number_format = '0.0'

# === KPI 1 ===
kpi1_row = 6
ws_tdb2[f'B{kpi1_row}'] = "Nombre de Dépressifs selon le genre"
ws_tdb2[f'B{kpi1_row}'].fill = kpi_title_fill
ws_tdb2[f'B{kpi1_row}'].font = kpi_title_font
ws_tdb2[f'B{kpi1_row}'].alignment = center_align
ws_tdb2[f'B{kpi1_row}'].border = border
ws_tdb2.merge_cells(f'B{kpi1_row}:E{kpi1_row}')
ws_tdb2.row_dimensions[kpi1_row].height = 20

kpi1_value_row = kpi1_row + 1
ws_tdb2[f'B{kpi1_value_row}'] = f'=TCD!B101'
ws_tdb2[f'B{kpi1_value_row}'].fill = kpi_male_fill
ws_tdb2[f'B{kpi1_value_row}'].font = kpi_value_font
ws_tdb2[f'B{kpi1_value_row}'].alignment = center_align
ws_tdb2[f'B{kpi1_value_row}'].border = border
ws_tdb2[f'B{kpi1_value_row}'].number_format = '0'
ws_tdb2.merge_cells(f'B{kpi1_value_row}:C{kpi1_value_row}')
ws_tdb2.row_dimensions[kpi1_value_row].height = 35

ws_tdb2[f'D{kpi1_value_row}'] = f'=TCD!B102'
ws_tdb2[f'D{kpi1_value_row}'].fill = kpi_female_fill
ws_tdb2[f'D{kpi1_value_row}'].font = kpi_value_font
ws_tdb2[f'D{kpi1_value_row}'].alignment = center_align
ws_tdb2[f'D{kpi1_value_row}'].border = border
ws_tdb2[f'D{kpi1_value_row}'].number_format = '0'
ws_tdb2.merge_cells(f'D{kpi1_value_row}:E{kpi1_value_row}')

# === KPI 2 ===
kpi2_row = 10
ws_tdb2[f'B{kpi2_row}'] = "Niveau Addiction moyen selon le genre"
ws_tdb2[f'B{kpi2_row}'].fill = kpi_title_fill
ws_tdb2[f'B{kpi2_row}'].font = kpi_title_font
ws_tdb2[f'B{kpi2_row}'].alignment = center_align
ws_tdb2[f'B{kpi2_row}'].border = border
ws_tdb2.merge_cells(f'B{kpi2_row}:E{kpi2_row}')
ws_tdb2.row_dimensions[kpi2_row].height = 20

kpi2_value_row = kpi2_row + 1
ws_tdb2[f'B{kpi2_value_row}'] = f'=TCD!B106'
ws_tdb2[f'B{kpi2_value_row}'].fill = kpi_male_fill
ws_tdb2[f'B{kpi2_value_row}'].font = kpi_value_font
ws_tdb2[f'B{kpi2_value_row}'].alignment = center_align
ws_tdb2[f'B{kpi2_value_row}'].border = border
ws_tdb2[f'B{kpi2_value_row}'].number_format = '0.0'
ws_tdb2.merge_cells(f'B{kpi2_value_row}:C{kpi2_value_row}')
ws_tdb2.row_dimensions[kpi2_value_row].height = 35

ws_tdb2[f'D{kpi2_value_row}'] = f'=TCD!B107'
ws_tdb2[f'D{kpi2_value_row}'].fill = kpi_female_fill
ws_tdb2[f'D{kpi2_value_row}'].font = kpi_value_font
ws_tdb2[f'D{kpi2_value_row}'].alignment = center_align
ws_tdb2[f'D{kpi2_value_row}'].border = border
ws_tdb2[f'D{kpi2_value_row}'].number_format = '0.0'
ws_tdb2.merge_cells(f'D{kpi2_value_row}:E{kpi2_value_row}')

print("\n✅ KPI TDB2 créés avec filtre Genre !")
print(f"   ✓ Réagissent maintenant au filtre Genre (TDB2!$G$3)\n")

wb.save(path_file)
wb.close()

print(f"✅ Fichier sauvegardé !\n")


✅ KPI TDB2 créés avec filtre Genre !
   ✓ Réagissent maintenant au filtre Genre (TDB2!$G$3)

✅ Fichier sauvegardé !



### Perf scolaire selon le genre et temps écran avant de dormir

In [1695]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.chart import BarChart, Reference
from openpyxl.chart.label import DataLabelList

wb = load_workbook(path_file)
ws_calc = wb["CALC"]
ws_tdb2 = wb["TDB2"]

# Réinitialiser TCD2
if "TCD2" in wb.sheetnames:
    del wb["TCD2"]
ws_tcd2 = wb.create_sheet("TCD2")

# Styles
title_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")
title_font = Font(bold=True, size=12, color="FFFFFF")
header_fill = PatternFill(start_color="FF008080", end_color="FF008080", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True, size=10)
row_header_fill = PatternFill(start_color="FFB3E5E0", end_color="FFB3E5E0", fill_type="solid")
row_header_font = Font(bold=True, color="000000", size=10)
border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
center_align = Alignment(horizontal="center", vertical="center")

# =================================================================
# TABLEAU 1 : Performance Scolaire (Lignes 1-8)
# =================================================================
start_row_perf = 1
ws_tcd2[f'A{start_row_perf}'] = "Performance Scolaire Moyenne par Sexe et Temps d'Écran"
ws_tcd2.merge_cells(f'A{start_row_perf}:M{start_row_perf}')
ws_tcd2[f'A{start_row_perf}'].font = title_font
ws_tcd2[f'A{start_row_perf}'].fill = title_fill
ws_tcd2[f'A{start_row_perf}'].alignment = center_align

header_row = 2
ws_tcd2[f'A{header_row}'] = "Sexe"
ws_tcd2[f'A{header_row}'].fill = header_fill
ws_tcd2[f'A{header_row}'].border = border

temps_cats = [("0-1h", 0, 1), ("1-2h", 1, 2), ("2-3h", 2, 3), ("3-4h", 3, 4)]
for i, (label, min_v, max_v) in enumerate(temps_cats):
    col = get_column_letter(i + 2)
    ws_tcd2[f'{col}{header_row}'] = label
    ws_tcd2[f'{col}{header_row}'].fill = header_fill
    ws_tcd2[f'{col}{header_row}'].border = border

for i, sexe in enumerate(["Male", "Female"]):
    row = 3 + i
    ws_tcd2[f'A{row}'] = sexe
    for j, (label, min_v, max_v) in enumerate(temps_cats):
        col = get_column_letter(j + 2)
        formula = (f'=IFERROR(AVERAGEIFS(DATA!$G:$G, DATA!$B:$B, "{sexe.lower()}", DATA!$F:$F, ">="&{min_v}, '
                   f'DATA!$F:$F, "<"&{max_v}, DATA!$A:$A, IF(TDB2!$D$3="Tous", "<>", TDB2!$D$3), '
                   f'DATA!$D:$D, IF(TDB2!$J$3="Tous", "<>", TDB2!$J$3), DATA!$I:$I, IF(TDB2!$M$3="Tous", "<>", TDB2!$M$3)), 0)')
        ws_tcd2[f'{col}{row}'] = formula
        ws_tcd2[f'{col}{row}'].number_format = '0.0'


# =================================================================
# TABLEAU 2 : Dépressifs par Âge (Lignes 10+)
# =================================================================
start_row = 10

# Styles
title_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")
title_font = Font(bold=True, size=12, color="FFFFFF")
header_fill = PatternFill(start_color="FF008080", end_color="FF008080", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True, size=10)
row_header_fill = PatternFill(start_color="FFB3E5E0", end_color="FFB3E5E0", fill_type="solid")
row_header_font = Font(bold=True, color="000000", size=10)
border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
center_align = Alignment(horizontal="center", vertical="center")


# --- TITRE ---
ws_tcd2[f'A{start_row}'] = "Nombre de dépressifs par Âge"
ws_tcd2[f'A{start_row}'].font = title_font
ws_tcd2[f'A{start_row}'].fill = title_fill
ws_tcd2[f'A{start_row}'].alignment = center_align
ws_tcd2.merge_cells(f'A{start_row}:B{start_row}')
ws_tcd2.row_dimensions[start_row].height = 25

# --- EN-TÊTES ---
header_row = start_row + 1

ws_tcd2[f'A{header_row}'] = "Âge"
ws_tcd2[f'A{header_row}'].fill = header_fill
ws_tcd2[f'A{header_row}'].font = header_font
ws_tcd2[f'A{header_row}'].border = border
ws_tcd2[f'A{header_row}'].alignment = center_align
ws_tcd2.column_dimensions['A'].width = 12

ws_tcd2[f'B{header_row}'] = "Nombre de dépressifs"
ws_tcd2[f'B{header_row}'].fill = header_fill
ws_tcd2[f'B{header_row}'].font = header_font
ws_tcd2[f'B{header_row}'].border = border
ws_tcd2[f'B{header_row}'].alignment = center_align
ws_tcd2.column_dimensions['B'].width = 18

# --- DONNÉES : Âges et Dépressifs ---
# Pour chaque âge possible (récupéré depuis CALC!C)
for age_idx in range(1, len_dict['len_age']):
    row_num = header_row + age_idx
    
    # Colonne A : Affiche l'âge seulement si :
    # - Filtre Âge = "Tous" (affiche tous les âges)
    # - OU l'âge = le filtre Âge
    ws_tcd2[f'A{row_num}'] = (
        f'=IF(OR(TDB2!$D$3="Tous", IFERROR(INDEX(CALC!$C$2:$C$100,{age_idx}),)=TDB2!$D$3), '
        f'IFERROR(INDEX(CALC!$C$2:$C$100,{age_idx}),""), "")'
    )
    ws_tcd2[f'A{row_num}'].fill = row_header_fill
    ws_tcd2[f'A{row_num}'].font = row_header_font
    ws_tcd2[f'A{row_num}'].border = border
    ws_tcd2[f'A{row_num}'].alignment = center_align
    
    # Colonne B : Compte dépressifs pour cet âge (si filtre = "Tous" ou = cet âge)
    # Avec filtres TDB2 : Genre, Plateforme, Interaction
    ws_tcd2[f'B{row_num}'] = (
        f'=IF(OR(TDB2!$D$3="Tous", IFERROR(INDEX(CALC!$C$2:$C$100,{age_idx}),)=TDB2!$D$3), '
        f'IFERROR(COUNTIFS('
        f'DATA!$M$2:$M$1201, 1, '  # depression_label = 1
        f'DATA!$A$2:$A$1201, IF(TDB2!$D$3="Tous", IFERROR(INDEX(CALC!$C$2:$C$100,{age_idx}),), TDB2!$D$3), '  # Age
        f'DATA!$B$2:$B$1201, IF(TDB2!$G$3="Tous", "<>", TDB2!$G$3), '  # Genre
        f'DATA!$D$2:$D$1201, IF(TDB2!$J$3="Tous", "<>", TDB2!$J$3), '  # Plateforme
        f'DATA!$I$2:$I$1201, IF(TDB2!$M$3="Tous", "<>", TDB2!$M$3)), 0), "")'
    )
    ws_tcd2[f'B{row_num}'].border = border
    ws_tcd2[f'B{row_num}'].alignment = center_align
    ws_tcd2[f'B{row_num}'].number_format = '0'

print("\n✅ Tableau TCD2 'Dépressifs par Âge' créé (DYNAMIQUE) !")
print(f"   📊 Comportement :")
print(f"      ✓ Filtre Âge='Tous' → Affiche tous les âges avec dépressifs")
print(f"      ✓ Filtre Âge='19' → Affiche seulement l'âge 19 avec ses dépressifs")
print(f"   ✓ Filtres appliqués : Genre, Plateforme, Interaction (TDB2)")
print(f"   ✓ Dynamique selon les filtres TDB2\n")

wb.save(path_file)
wb.close()

print(f"✅ Fichier sauvegardé !\n")


✅ Tableau TCD2 'Dépressifs par Âge' créé (DYNAMIQUE) !
   📊 Comportement :
      ✓ Filtre Âge='Tous' → Affiche tous les âges avec dépressifs
      ✓ Filtre Âge='19' → Affiche seulement l'âge 19 avec ses dépressifs
   ✓ Filtres appliqués : Genre, Plateforme, Interaction (TDB2)
   ✓ Dynamique selon les filtres TDB2

✅ Fichier sauvegardé !



In [1696]:
from openpyxl import load_workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.chart.label import DataLabelList

wb = load_workbook(path_file)
ws_tcd2 = wb["TCD2"]
ws_tdb2 = wb["TDB2"]

# =================================================================
# GRAPHIQUE : Performance Scolaire par Temps d'Écran et Sexe
# =================================================================

chart = BarChart()
chart.type = "col"
chart.style = 10
chart.title = "Performance scolaire moyenne par temps d'écran"
chart.y_axis.title = "Performance scolaire"
chart.x_axis.title = "Temps d'écran (Heures)"
chart.height = 7 
chart.width = 10

# 1. Références de données
# min_col=1 (colonne A contenant "Male"/"Female")
# max_col=5 (colonne E contenant les données de la 4ème catégorie)
# min_row=3, max_row=4 (les deux lignes de données)
data = Reference(ws_tcd2, min_col=1, min_row=3, max_col=5, max_row=4)

# Les catégories sont sur la ligne 2, de la colonne B à E
cats = Reference(ws_tcd2, min_col=2, min_row=2, max_col=5)

# from_rows=True est obligatoire ici car chaque ligne est une série (Male/Female)
chart.add_data(data, titles_from_data=True, from_rows=True)
chart.set_categories(cats)
chart.overlap = -15

# 2. Configuration des étiquettes
for series in chart.series:
    series.dLbls = DataLabelList()
    series.dLbls.showVal = True
    series.dLbls.position = 'outEnd'
    series.dLbls.numFmt = "0.00"

# 3. Application des couleurs (Série 0 = Male, Série 1 = Female)
colors = ["040459", "7d093f"]

for i, series in enumerate(chart.series):
    if i < len(colors):
        series.graphicalProperties.solidFill = colors[i]
        for point in series.data_points:
            point.graphicalProperties.solidFill = colors[i]

chart.legend.position = "r"
chart.y_axis.majorGridlines = None

# 4. Placement sur TDB2
ws_tdb2.add_chart(chart, "G5")

wb.save(path_file)
wb.close()

print("✅ Graphique de performance scolaire mis à jour avec succès.")

✅ Graphique de performance scolaire mis à jour avec succès.


In [1697]:
from openpyxl.chart import BarChart, Reference
from openpyxl.chart.label import DataLabelList

# =================================================================
# BARCHART : Dépressifs par Âge (TDB2)
# =================================================================

start_row_dep = 10  # Où commence le tableau des dépressifs
header_row_dep = start_row_dep + 1  # Ligne 11 (en-têtes)
last_row_dep = header_row_dep + len_dict['len_age'] - 1  # Dernière ligne de données

chart_dep = BarChart()
chart_dep.type = "bar"  # Horizontal
chart_dep.title = "Nombre de dépressifs par Âge"
chart_dep.height = 7
chart_dep.width = 7

# Références
# data : colonne B (Nombre de dépressifs) avec en-tête
# cats : colonne A (Âges) sans en-tête
data = Reference(
    ws_tcd2,
    min_col=2,
    min_row=header_row_dep,  # Ligne 11 (en-tête "Nombre de dépressifs")
    max_row=last_row_dep
)

cats = Reference(
    ws_tcd2,
    min_col=1,
    min_row=header_row_dep + 1,  # Ligne 12 (première donnée d'âge)
    max_row=last_row_dep
)

chart_dep.add_data(data, titles_from_data=True)
chart_dep.set_categories(cats)

# Style
chart_dep.dataLabels = DataLabelList()
chart_dep.dataLabels.showVal = True
chart_dep.dataLabels.position = 'outEnd'
chart_dep.series[0].graphicalProperties.solidFill = "7D093F"  # Rose foncé pour dépression

chart_dep.legend.position = "r"
chart_dep.y_axis.majorGridlines = None

# Ajout à TDB2 (position à côté du tableau)
ws_tdb2.add_chart(chart_dep, "L5")

wb.save(path_file)
wb.close()

In [1698]:
from openpyxl.chart import ScatterChart, Reference, Series
from openpyxl.chart.label import DataLabelList


wb = load_workbook(path_file)

# Créer les feuilles

if "TCD3" in wb.sheetnames:
    del wb["TCD3"]
ws_tcd3 = wb.create_sheet("TCD3")

if "TDB3" in wb.sheetnames:
    del wb["TDB3"]

ws_tdb3 = wb.create_sheet("TDB3")
ws_data = wb["DATA"]

title_fill_main = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type='solid')
title_font_main = Font(name='Calibri', size=14, bold=True, color='FFFFFF')
title_alignment_main = Alignment(horizontal='center', vertical='center', wrap_text=True)

ws_tdb3.merge_cells('A1:O1')
ws_tdb3['A1'] = 'Analyse des corrélations avancées (Santé Mentale)'
ws_tdb3['A1'].fill = title_fill_main
ws_tdb3['A1'].font = title_font_main
ws_tdb3['A1'].alignment = title_alignment_main
ws_tdb3.row_dimensions[1].height = 30
ws_tdb3.sheet_view.showGridLines = False

# Styles

title_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")

title_font = Font(bold=True, size=12, color="FFFFFF")

header_fill = PatternFill(start_color="FF008080", end_color="FF008080", fill_type="solid")

header_font = Font(color="FFFFFF", bold=True, size=10)

border = Border(left=Side(style='thin'), right=Side(style='thin'),

                top=Side(style='thin'), bottom=Side(style='thin'))

center_align = Alignment(horizontal="center", vertical="center")



def style_header(ws, row, cols=['A', 'B', 'C']):

    for col in cols:

        ws[f'{col}{row}'].fill = header_fill

        ws[f'{col}{row}'].font = header_font

        ws[f'{col}{row}'].border = border

        ws[f'{col}{row}'].alignment = center_align



def style_title(ws, row, text, span='A:C'):

    start, end = span.split(':')

    ws[f'{start}{row}'] = text

    ws[f'{start}{row}'].font = title_font

    ws[f'{start}{row}'].fill = title_fill

    ws.merge_cells(f'{start}{row}:{end}{row}')

    ws.row_dimensions[row].height = 25


# ================================================================
# TABLEAU 1 : Sommeil vs Addiction (lignes 1-12)
# ================================================================

style_title(ws_tcd3, 1, "Sommeil vs Addiction par Interaction Sociale")
ws_tcd3['A2'] = "Sommeil (h)"
ws_tcd3['B2'] = "Addiction (Low)"
ws_tcd3['C2'] = "Addiction (High)"
style_header(ws_tcd3, 2)

for idx, sleep_val in enumerate([4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5]):
    row_num = 3 + idx
    ws_tcd3[f'A{row_num}'] = sleep_val
    ws_tcd3[f'A{row_num}'].number_format = '0.0'
    ws_tcd3[f'A{row_num}'].alignment = center_align

    # ✅ Renvoie 0 (pas "") pour que le ScatterChart trace les points
    ws_tcd3[f'B{row_num}'] = (
        f'=IFERROR(AVERAGEIFS(DATA!$L$2:$L$1201, DATA!$E$2:$E$1201, ">="&({sleep_val}-0.25), '
        f'DATA!$E$2:$E$1201, "<"&({sleep_val}+0.25), DATA!$I$2:$I$1201, "low"), 0)'
    )
    ws_tcd3[f'B{row_num}'].number_format = '0.00'
    ws_tcd3[f'B{row_num}'].alignment = center_align

    ws_tcd3[f'C{row_num}'] = (
        f'=IFERROR(AVERAGEIFS(DATA!$L$2:$L$1201, DATA!$E$2:$E$1201, ">="&({sleep_val}-0.25), '
        f'DATA!$E$2:$E$1201, "<"&({sleep_val}+0.25), DATA!$I$2:$I$1201, "high"), 0)'
    )
    ws_tcd3[f'C{row_num}'].number_format = '0.00'
    ws_tcd3[f'C{row_num}'].alignment = center_align

    for col in ['A', 'B', 'C']:
        ws_tcd3[f'{col}{row_num}'].border = border

# ================================================================
# TABLEAU 2 : Performance vs Stress (lignes 15-26)
# ================================================================

style_title(ws_tcd3, 15, "Performance vs Stress par Genre")
ws_tcd3['A16'] = "Stress"
ws_tcd3['B16'] = "Performance (Male)"
ws_tcd3['C16'] = "Performance (Female)"
style_header(ws_tcd3, 16)

for stress_val in range(1, 11):
    row_num = 16 + stress_val
    ws_tcd3[f'A{row_num}'] = stress_val
    ws_tcd3[f'A{row_num}'].alignment = center_align

    ws_tcd3[f'B{row_num}'] = (
        f'=IFERROR(AVERAGEIFS(DATA!$G$2:$G$1201, DATA!$J$2:$J$1201, {stress_val}, '
        f'DATA!$B$2:$B$1201, "male"), 0)'
    )
    ws_tcd3[f'B{row_num}'].number_format = '0.00'
    ws_tcd3[f'B{row_num}'].alignment = center_align

    ws_tcd3[f'C{row_num}'] = (
        f'=IFERROR(AVERAGEIFS(DATA!$G$2:$G$1201, DATA!$J$2:$J$1201, {stress_val}, '
        f'DATA!$B$2:$B$1201, "female"), 0)'
    )
    ws_tcd3[f'C{row_num}'].number_format = '0.00'
    ws_tcd3[f'C{row_num}'].alignment = center_align

    for col in ['A', 'B', 'C']:
        ws_tcd3[f'{col}{row_num}'].border = border

# ================================================================
# TABLEAU 3 : Anxiété vs Temps d'écran (lignes 30-39)
# ================================================================

style_title(ws_tcd3, 30, "Anxiété vs Temps d'Écran par Interaction")
ws_tcd3['A31'] = "Temps d'écran (h)"
ws_tcd3['B31'] = "Anxiété (Low)"
ws_tcd3['C31'] = "Anxiété (High)"
style_header(ws_tcd3, 31)

for idx, screen_val in enumerate([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5]):
    row_num = 32 + idx
    ws_tcd3[f'A{row_num}'] = screen_val
    ws_tcd3[f'A{row_num}'].number_format = '0.0'
    ws_tcd3[f'A{row_num}'].alignment = center_align

    ws_tcd3[f'B{row_num}'] = (
        f'=IFERROR(AVERAGEIFS(DATA!$K$2:$K$1201, DATA!$F$2:$F$1201, ">="&({screen_val}-0.25), '
        f'DATA!$F$2:$F$1201, "<"&({screen_val}+0.25), DATA!$I$2:$I$1201, "low"), 0)'
    )
    ws_tcd3[f'B{row_num}'].number_format = '0.00'
    ws_tcd3[f'B{row_num}'].alignment = center_align

    ws_tcd3[f'C{row_num}'] = (
        f'=IFERROR(AVERAGEIFS(DATA!$K$2:$K$1201, DATA!$F$2:$F$1201, ">="&({screen_val}-0.25), '
        f'DATA!$F$2:$F$1201, "<"&({screen_val}+0.25), DATA!$I$2:$I$1201, "high"), 0)'
    )
    ws_tcd3[f'C{row_num}'].number_format = '0.00'
    ws_tcd3[f'C{row_num}'].alignment = center_align

    for col in ['A', 'B', 'C']:
        ws_tcd3[f'{col}{row_num}'].border = border

# ================================================================
# TABLEAU 4 : Performance vs Sommeil (lignes 42-53)
# ================================================================

style_title(ws_tcd3, 42, "Performance vs Sommeil par État Dépressif")
ws_tcd3['A43'] = "Sommeil (h)"
ws_tcd3['B43'] = "Performance (No Dep)"
ws_tcd3['C43'] = "Performance (Dep)"
style_header(ws_tcd3, 43)

for idx, sleep_val in enumerate([4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5]):
    row_num = 44 + idx
    ws_tcd3[f'A{row_num}'] = sleep_val
    ws_tcd3[f'A{row_num}'].number_format = '0.0'
    ws_tcd3[f'A{row_num}'].alignment = center_align

    ws_tcd3[f'B{row_num}'] = (
        f'=IFERROR(AVERAGEIFS(DATA!$G$2:$G$1201, DATA!$E$2:$E$1201, ">="&({sleep_val}-0.25), '
        f'DATA!$E$2:$E$1201, "<"&({sleep_val}+0.25), DATA!$M$2:$M$1201, 0), 0)'
    )
    ws_tcd3[f'B{row_num}'].number_format = '0.00'
    ws_tcd3[f'B{row_num}'].alignment = center_align

    ws_tcd3[f'C{row_num}'] = (
        f'=IFERROR(AVERAGEIFS(DATA!$G$2:$G$1201, DATA!$E$2:$E$1201, ">="&({sleep_val}-0.25), '
        f'DATA!$E$2:$E$1201, "<"&({sleep_val}+0.25), DATA!$M$2:$M$1201, 1), 0)'
    )
    ws_tcd3[f'C{row_num}'].number_format = '0.00'
    ws_tcd3[f'C{row_num}'].alignment = center_align

    for col in ['A', 'B', 'C']:
        ws_tcd3[f'{col}{row_num}'].border = border

# Largeurs colonnes
ws_tcd3.column_dimensions['A'].width = 16
ws_tcd3.column_dimensions['B'].width = 18
ws_tcd3.column_dimensions['C'].width = 18


# ================================================================
# FONCTION : créer un ScatterChart avec Series X/Y explicites
# ================================================================

def make_scatter(title, x_title, y_title, data_min_row, data_max_row,
                 serie_labels, serie_colors, x_header_row):
    """
    Crée un nuage de points où :
    - Colonne A = valeurs X (xvalues)
    - Colonnes B et C = valeurs Y (yvalues)
    - Les en-têtes (row x_header_row) servent de titre de série
    """
    chart = ScatterChart()
    chart.title = title
    chart.x_axis.title = x_title
    chart.y_axis.title = y_title
    chart.height = 9
    chart.width = 14
    chart.style = 13

    # ✅ Axe X numérique (essentiel pour scatter)
    chart.x_axis.delete = False
    chart.y_axis.delete = False

    # X = colonne A (commune aux deux séries)
    xvalues = Reference(ws_tcd3, min_col=1, min_row=data_min_row, max_row=data_max_row)

    for i, (label, color) in enumerate(zip(serie_labels, serie_colors)):
        col = 2 + i  # B puis C
        # Y inclut l'en-tête pour récupérer le titre via from_rows
        yvalues = Reference(ws_tcd3, min_col=col, min_row=x_header_row, max_row=data_max_row)
        serie = Series(yvalues, xvalues, title_from_data=True)
        serie.marker.symbol = "circle"
        serie.marker.size = 8
        serie.graphicalProperties.line.noFill = True  # points seuls, pas de ligne
        serie.marker.graphicalProperties.solidFill = color
        serie.marker.graphicalProperties.line.solidFill = color
        chart.series.append(serie)

    chart.legend.position = "r"
    chart.y_axis.majorGridlines = None
    return chart


# ================================================================
# CRÉATION DES 4 SCATTER PLOTS
# ================================================================

# Graphique 1 : Sommeil vs Addiction
chart1 = make_scatter(
    "Sommeil vs Addiction par Interaction Sociale",
    "Temps de Sommeil (heures)", "Niveau d'Addiction",
    data_min_row=3, data_max_row=12,
    serie_labels=["Intéraction faible", "Intéraction élevée"],
    serie_colors=["20B2AA", "FF6B7B"],
    x_header_row=2
)
ws_tdb3.add_chart(chart1, "B4")

# Graphique 2 : Performance vs Stress
chart2 = make_scatter(
    "Performance Scolaire selon Stress par Genre",
    "Niveau de Stress", "Performance Scolaire",
    data_min_row=17, data_max_row=26,
    serie_labels=["Performance (Male)", "Performance (Female)"],
    serie_colors=["4169E1", "FF1493"],
    x_header_row=16
)
ws_tdb3.add_chart(chart2, "K4")

# Graphique 3 : Anxiété vs Temps d'écran
chart3 = make_scatter(
    "Anxiété selon Temps d'Écran par Interaction sociale",
    "Temps d'Écran avant Sommeil (heures)", "Niveau d'Anxiété",
    data_min_row=32, data_max_row=38,
    serie_labels=["Anxiété (Low)", "Anxiété (High)"],
    serie_colors=["20B2AA", "FF6B7B"],
    x_header_row=31
)
ws_tdb3.add_chart(chart3, "B24")

# Graphique 4 : Performance vs Sommeil
chart4 = make_scatter(
    "Performance Scolaire selon Sommeil par État Dépressif",
    "Temps de Sommeil (heures)", "Performance Scolaire",
    data_min_row=44, data_max_row=53,
    serie_labels=["Non dépressive", "Dépressive"],
    serie_colors=["20B2AA", "FF6B7B"],
    x_header_row=43
)
ws_tdb3.add_chart(chart4, "K24")

print("\n✅ TDB3 et TCD3 créées avec 4 nuages de points fonctionnels !")
print("   📊 Points X = colonne A, Y = colonnes B et C")
print("   📈 Lignes désactivées (points seuls)\n")

wb.save(path_file)
wb.close()
print("✅ Fichier sauvegardé !\n")


✅ TDB3 et TCD3 créées avec 4 nuages de points fonctionnels !
   📊 Points X = colonne A, Y = colonnes B et C
   📈 Lignes désactivées (points seuls)

✅ Fichier sauvegardé !



In [1699]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.utils import get_column_letter

# 1. Chargement du fichier
wb = load_workbook(path_file)
ws_corr = wb["Correlations"]

# Nettoyer et créer TDB4
if "TDB4" in wb.sheetnames:
    del wb["TDB4"]
TDB4 = wb.create_sheet("TDB4", 3)
TDB4.sheet_view.showGridLines = False

# 2. Styles de la charte graphique
title_fill_main = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type='solid')
title_font_main = Font(name='Calibri', size=14, bold=True, color='FFFFFF')
header_fill = PatternFill(start_color="FF008080", end_color="FF008080", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True, size=10)
border_thin = Border(left=Side(style='thin', color='DDDDDD'), right=Side(style='thin', color='DDDDDD'),
                     top=Side(style='thin', color='DDDDDD'), bottom=Side(style='thin', color='DDDDDD'))
center_align = Alignment(horizontal='center', vertical='center')

# ========== TITRE PRINCIPAL BLEU FONCÉ ==========
TDB3_columns_count = 11  # 1 colonne labels + 10 colonnes variables
last_col_letter = get_column_letter(TDB3_columns_count + 1)

TDB4.merge_cells(f'A1:{last_col_letter}1')
TDB4['A1'] = 'Tableau de Bord 4 : Carte des Corrélations de Santé Mentale'
TDB4['A1'].fill = title_fill_main
TDB4['A1'].font = title_font_main
TDB4['A1'].alignment = title_alignment_main = Alignment(horizontal='center', vertical='center')
TDB4.row_dimensions[1].height = 35

# ========== PRÉPARATION DE LA MATRICE VISUELLE ==========
start_row_matrix = 4
cols_corr = ["age", "sleep_hours", "daily_social_media_hours",
             "academic_performance", "physical_activity",
             "social_num", "stress_level", "anxiety_level",
             "addiction_level", "depression_label"]

# Écriture des en-têtes de colonnes
for col_idx, var_name in enumerate(cols_corr):
    col_letter = get_column_letter(col_idx + 2)
    TDB4[f'{col_letter}{start_row_matrix}'] = var_name
    TDB4[f'{col_letter}{start_row_matrix}'].fill = header_fill
    TDB4[f'{col_letter}{start_row_matrix}'].font = header_font
    TDB4[f'{col_letter}{start_row_matrix}'].alignment = center_align
    TDB4[f'{col_letter}{start_row_matrix}'].border = border_thin

# Écriture des lignes avec liaisons dynamiques vers la feuille 'Correlations'
for row_idx, var_name in enumerate(cols_corr):
    current_row = start_row_matrix + 1 + row_idx
    TDB4.row_dimensions[current_row].height = 20
    
    # Label de ligne (colonne A)
    TDB4[f'A{current_row}'] = var_name
    TDB4[f'A{current_row}'].font = Font(bold=True, size=10)
    TDB4[f'A{current_row}'].border = border_thin
    
    for col_idx in range(len(cols_corr)):
        col_letter = get_column_letter(col_idx + 2)
        
        # Liaison dynamique vers les données calculées dans la feuille Correlations
        # Le tableau dans Correlations commence en ligne 3 (ligne 1=titre, ligne 2=headers)
        corr_row = 3 + row_idx
        corr_col_letter = get_column_letter(2 + col_idx)
        
        TDB4[f'{col_letter}{current_row}'] = f"=Correlations!{corr_col_letter}{corr_row}"
        TDB4[f'{col_letter}{current_row}'].number_format = '0.00'
        TDB4[f'{col_letter}{current_row}'].alignment = center_align
        TDB4[f'{col_letter}{current_row}'].border = border_thin

# Auto-ajustement de la largeur des colonnes pour la lisibilité
TDB4.column_dimensions['A'].width = 24
for col_idx in range(len(cols_corr)):
    col_letter = get_column_letter(col_idx + 2)
    TDB4.column_dimensions[col_letter].width = 22

# ========== APPLICATION DE LA CARTE DE CHALEUR (HEATMAP) ==========
# Règle de dégradé à 3 couleurs : 
# - Corrélation négative (-1.00) 
# - Neutre (0.00)
# - Corrélation positive (1.00)
color_scale = ColorScaleRule(
    start_type='num', start_value=-1.0, start_color='FFF2F2',  # Rouge très clair (Négatif)
    mid_type='num', mid_value=0.0, mid_color='FFFFFF',        # Blanc (Aucune relation)
    end_type='num', end_value=1.0, end_color='E6F2FF'          # Bleu très clair (Positif)
)

# Plage de la matrice de données : de B5 à K14
matrix_range = f"B5:{get_column_letter(len(cols_corr) + 1)}{start_row_matrix + len(cols_corr)}"
TDB4.conditional_formatting.add(matrix_range, color_scale)

# Sauvegarde
wb.save(path_file)
wb.close()
print("✅ Feuille TDB4 créée avec la matrice de corrélation au format Heatmap !")

✅ Feuille TDB4 créée avec la matrice de corrélation au format Heatmap !
